# Preprocessing and data-quality audit: `all_removals`

A **read-only** audit of the existing warehouse, done before any model-specific preparation. It answers: what is in the data, how the joins expand it, where values or links look wrong, which observations repeat, how much each *proposed* rule would remove, and how unevenly the data is spread.

**Ground rules this notebook follows**

- Nothing in the ETL, the warehouse Parquet files or `warehouse.duckdb` is changed or rerun. The notebook opens an **in-memory** DuckDB database whose views point at the existing Parquet files, and rebuilds the `build_views.py` eligibility views from the same SQL. `warehouse.duckdb` is never opened, so it can't lock out Slurm jobs.
- Every derived file goes to `data/processed/preprocessing_audit/<case_study>/` (git-ignored). Nothing is written next to the warehouse tables.
- Three kinds of result are kept apart, and each table says which it is:
  - **Flag**: a diagnostic label on an observation, pair or station. Nothing is removed or corrected.
  - **Hypothetical rule**: "if we excluded X, this many rows, stations, pairs and dams would remain". Counted, never applied.
  - **Derived feature**: a new column computed from existing data (log distance, drainage-area ratio, backwater screen, robust z). Saved to new files only; the source values stay untouched.

**Sections**

| | Section | Main outputs |
|---|---|---|
| A | Data inventory and pipeline audit | tables, schemas, join expansion, multi-dam reuse, spatial linkage |
| B | Measurement quality and units | units, fractions, methods, zeros/bounds/extremes, unit-error candidates, cross-parameter consistency |
| C | Duplicates and repeated measurements | duplicate reports, co-located stations, same-day readings, reuse across links |
| D | Robust outlier flags | within-station MAD flags, log vs linear scale, threshold sensitivity |
| E | Study eligibility and covariate coverage | site type, drainage area, snapping, coverage, temporal coverage, hypothetical minimum-data rules |
| F | Overlapping exposures and upstream stations | overlapping removal windows, hypothetical assignment rules, backwater screen |
| G | Distance and drainage-area features | log distance, dam drainage area, station/dam area ratio |
| H | Sampling imbalance | counts by dimension, concentration, row- vs pair- vs dam-weighted shares |
| I | Stepwise audit table and findings | methods table, findings summary |

**How to run.** On the cluster, on a compute node (not the login node), with the `dam-removal` environment: open it in Jupyter, or run
`jupyter nbconvert --to html --execute --ExecutePreprocessor.timeout=-1 notebooks/preprocessing_audit.ipynb`.
Ask for a few cores and ~32 GB of memory (DuckDB spills to `OUTPUT_DIR/_duckdb_tmp` if it needs more). The slow steps (reading `wq_measurement_raw` for method fields, and the NHDPlus attributes) have switches in the configuration cell; the NHDPlus step is cached after its first run.

## 0. Setup

Paths follow the other notebooks (`ROOT` is the repo; the case study comes from its config). Thresholds are all in one place below, so a rerun with different values is a one-line change. They are **exploratory starting points**, not decisions.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import duckdb

from analysis import audit_utils as au
from etl.build_views import DEFAULT_MIN_SAMPLES, DEFAULT_MIN_YEARS
from etl.common import wq_rules
from etl.common.config import load_config
from etl.common.nhdplus import find_huc8_geodatabase
from etl.common.warehouse import warehouse_dir

# Where data/ lives: the repo itself on the cluster. DAM_REMOVAL_PROJECT_ROOT
# (the ETL's own override) points it elsewhere, e.g. at a synthetic test copy.
DATA_ROOT = Path(os.environ.get("DAM_REMOVAL_PROJECT_ROOT", ROOT))

CASE_STUDY = "all_removals"
cfg = load_config(CASE_STUDY, project_root=DATA_ROOT)
WAREHOUSE = warehouse_dir(cfg)
NHDPLUS_DIR = DATA_ROOT / "data" / "processed" / "nhdplus_extracted"
# Every derived file goes here, never into the warehouse folder.
OUTPUT_DIR = DATA_ROOT / "data" / "processed" / "preprocessing_audit" / CASE_STUDY

window = cfg.get("wq_window", {}) or {}
YEARS_BEFORE = int(window.get("years_before", 3))
YEARS_AFTER = int(window.get("years_after", 3))
eligibility = cfg.get("eligibility", {}) or {}
MIN_SAMPLES = int(eligibility.get("min_samples", DEFAULT_MIN_SAMPLES))
MIN_YEARS = int(eligibility.get("min_years", DEFAULT_MIN_YEARS))

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

print("Case study:", cfg.name)
print("Warehouse (read only):", WAREHOUSE)
print("Audit outputs:", OUTPUT_DIR)
print(f"Window: {YEARS_BEFORE} years before / {YEARS_AFTER} after removal; "
      f"current eligibility: >= {MIN_SAMPLES} samples in >= {MIN_YEARS} years per side")

In [ ]:
# ---- Switches for the slower steps ----
SAVE_OUTPUTS = True        # write derived tables/flags to OUTPUT_DIR
RUN_METHOD_AUDIT = True    # read method/speciation fields from wq_measurement_raw (~10M rows)
RUN_NHDPLUS_VAA = True     # read NHDPlusFlowlineVAA for dam/station reaches (cached after first run)
REFRESH_VAA_CACHE = False  # re-read the NHDPlus packages even if the cache exists
MEMORY_LIMIT = None        # e.g. "48GB"; None = DuckDB default (80% of the node's RAM)

# ---- Exploratory thresholds (starting points, not decisions) ----
THRESHOLDS = {
    # D. Robust outliers: |robust z| above this is flagged, within station x series.
    "mad_threshold": 5.0,
    "mad_sensitivity": [3.5, 5.0, 7.0, 10.0],
    "mad_min_n": 10,                      # fewer reference values -> "sparse", not screened
    # A/E. Station snapping.
    "snap_distance_m": [100, 250, 500, 1000, 5000],
    "snap_rule_m": 500,                   # used by the hypothetical rule in section I
    # E. Possibly brackish/saline: station median specific conductance (uS/cm).
    "saline_sc_us_cm": 3000,
    # C. Co-located stations: same NHDPlus reach and within this distance (m).
    "colocated_max_m": 200,
    "duplicate_rel_tol": 0.005,           # values this close count as "the same"
    # B. Cross-parameter consistency (same sampling event).
    "consistency_rel_tol": 0.10,          # part may exceed whole by 10%...
    "consistency_abs_tol": {"phosphorus": 0.005, "orthophosphate": 0.005,
                            "nitrate": 0.05, "ammonia": 0.01},  # ...and by this much (mg/L)
    "tds_sc_ratio": [0.40, 1.00],         # TDS / specific conductance outside this range
    "do_saturation_tol_pct": 15,          # reported vs computed DO saturation (percentage points)
    # B. Unit-error candidates.
    "unit_factor_tolerance": {"x1000": 0.5, "basis": 0.15},  # x1000 slips: ratio within 50%;
                                          # N/P/NO3/PO4/NH4 basis factors: within 15%
    "unit_block_min_n": 3,
    # E. Series and minimum-data rules.
    "min_pairs_per_series": 20,
    "min_data_rules": [
        {"rule": "current: >=3 samples, >=2 years per side", "min_samples": 3, "min_years": 2},
        {"rule": ">=6 samples, >=2 years per side", "min_samples": 6, "min_years": 2},
        {"rule": ">=12 samples, >=3 years per side", "min_samples": 12, "min_years": 3},
        {"rule": ">=3 dates per side, >=3 shared months", "min_dates": 3, "min_years": 2,
         "min_shared_months": 3},
        {"rule": ">=6 dates per side, >=4 shared months", "min_dates": 6, "min_years": 2,
         "min_shared_months": 4},
        {"rule": ">=3 samples per side + data in years -2..-1 and +1..+2", "min_samples": 3,
         "min_years": 2, "near_removal": True},
    ],
    "rule_min_data": ">=6 samples, >=2 years per side",   # used in section I
    "rule_min_shared_months": 3,                         # used in section I
    # F. Overlap: another removal counts as "far" if this many times farther than the pair's dam.
    "overlap_distance_multiple": 3.0,
    # G. log10(distance + offset): keeps 0 km defined.
    "distance_log_offset_km": 0.1,
}

# Positive, right-skewed parameters: screened on log10 (as in analysis_eda.ipynb).
LOG_PARAMETERS = ["turbidity", "specific_conductance", "nitrate", "ammonia",
                  "orthophosphate", "phosphorus", "tds"]

# B. Exploratory "suspicious" limits, much tighter than wq_rules.PLAUSIBLE_RANGES
# (which only drops impossible values). Values beyond them are flagged, not removed.
SUSPICIOUS_LIMITS = {
    "water_temp": (None, 35.0), "ph": (4.0, 11.0), "dissolved_oxygen": (None, 20.0),
    "do_saturation": (None, 200.0), "specific_conductance": (None, 10000.0),
    "turbidity": (None, 2000.0), "phosphorus": (None, 10.0), "orthophosphate": (None, 5.0),
    "nitrate": (None, 30.0), "ammonia": (None, 10.0), "tds": (None, 10000.0),
}
# What an exact zero means per parameter.
ZERO_RULES = {
    "ph": "implausible", "specific_conductance": "implausible", "tds": "implausible",
    "dissolved_oxygen": "rare", "do_saturation": "rare", "turbidity": "rare",
    "nitrate": "should_be_censored", "ammonia": "should_be_censored",
    "orthophosphate": "should_be_censored", "phosphorus": "should_be_censored",
    "water_temp": "plausible",
}

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
con, VIEWS = au.connect(
    WAREHOUSE, YEARS_BEFORE, YEARS_AFTER, MIN_SAMPLES, MIN_YEARS,
    threads=os.environ.get("SLURM_CPUS_PER_TASK"), memory_limit=MEMORY_LIMIT,
    temp_dir=OUTPUT_DIR / "_duckdb_tmp",
)
HAS = set(VIEWS)
SAVED = []
TIMINGS = {}


def q(sql):
    """Run SQL on the in-memory audit database and return a DataFrame."""
    return con.sql(sql).df()


def save(frame, name):
    """Write a derived table to OUTPUT_DIR (only there) if SAVE_OUTPUTS."""
    if SAVE_OUTPUTS:
        au.save_output(frame, name, OUTPUT_DIR)
        SAVED.append(name)
    return frame


def sql_list(values):
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)


SERIES = au.series_sql()
LOG_LIST = sql_list(LOG_PARAMETERS)
print("Warehouse views available:", ", ".join(VIEWS))
missing = [t for t in ["analysis_measurement", "station_watershed", "climate", "land_cover", "streams"]
           if t not in HAS]
print("Missing optional tables:", missing or "none")
assert "analysis_measurement" in HAS, "analysis_measurement.parquet is needed for sections D-I"

### Working tables

Three temp tables (in memory only) used throughout:

- `m`: every cleaned result in `wq_measurement`, plus a `series` label (parameter × fraction, e.g. `phosphorus (Total)`) and the month.
- `am`: every row of `analysis_measurement` (one per measurement × eligible dam-station pair), plus `series` and a `pair` label.
- `pairs`: one row per dam-station pair in `am`.

In [ ]:
t0 = time.time()
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE m AS
    SELECT *, {SERIES} AS series, month(sample_date) AS month
    FROM wq_measurement
""")
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE am AS
    SELECT *, {SERIES} AS series, AR_ID || ' | ' || station_id AS pair
    FROM analysis_measurement
""")
con.execute("""
    CREATE OR REPLACE TEMP TABLE pairs AS
    SELECT AR_ID, station_id, pair,
           any_value(side) AS side, any_value(relationship) AS relationship,
           any_value(network_distance_km) AS network_distance_km,
           any_value(removal_year) AS removal_year,
           count(*) AS rows, count(DISTINCT series) AS n_series
    FROM am GROUP BY AR_ID, station_id, pair
""")
TIMINGS["working tables"] = time.time() - t0
print(q("SELECT (SELECT count(*) FROM m) AS wq_measurement_rows, (SELECT count(*) FROM am) AS analysis_rows, "
        "(SELECT count(*) FROM pairs) AS pairs").to_string(index=False))

## A. Data inventory and pipeline audit

### A.1 Source tables

Every warehouse Parquet file the audit reads (row counts come from Parquet metadata, so this is fast). The eligibility views (`measurement_window`, `eligibility_pairs`, `eligibility_dams`, `eligibility_summary`) exist only in this notebook's in-memory database, built from the same SQL as `etl/build_views.py`.

In [ ]:
inventory = pd.DataFrame([{
    "table": view,
    "rows": con.sql(f"SELECT count(*) FROM {view}").fetchone()[0],
    "columns": len(con.sql(f"SELECT * FROM {view} LIMIT 0").columns),
    "size_mb": round((WAREHOUSE / f"{view}.parquet").stat().st_size / 1e6, 1),
    "modified": datetime.fromtimestamp((WAREHOUSE / f"{view}.parquet").stat().st_mtime).strftime("%Y-%m-%d %H:%M"),
} for view in VIEWS])
save(inventory, "A1_inventory_tables.csv")
inventory

In [ ]:
# Column names and types of every table (saved in full; only a count is shown here).
schema = pd.concat([
    con.sql(f"DESCRIBE {view}").df().assign(table=view)[["table", "column_name", "column_type"]]
    for view in VIEWS
], ignore_index=True)
save(schema, "A1_inventory_schema.csv")
schema.groupby("table").size().rename("columns").to_frame().T

### A.2 Missingness and distinct values

`SUMMARIZE` per table: share of nulls and approximate distinct count for each column (geometry left out). **How to read it:** identifiers (`measurement_id`, `station_id`, `AR_ID`) should have 0% nulls; the context columns of `analysis_measurement` are null by design when the covariate is missing (spec 3.11, choice 4). Only columns with any nulls, plus key identifiers, are displayed; the full table is saved.

In [ ]:
PROFILE_TABLES = ["dam_removal", "wq_station", "dam_station_link", "wq_measurement",
                  "station_watershed", "climate", "analysis_measurement"]
KEY_COLUMNS = {"measurement_id", "station_id", "AR_ID", "parameter_std", "sample_date",
               "result_value_std", "result_unit_std", "NHDPlusID"}
profiles = []
for view in [t for t in PROFILE_TABLES if t in HAS]:
    cols = [c for c in con.sql(f"SELECT * FROM {view} LIMIT 0").columns if c != "geometry"]
    select = ", ".join(f'"{c}"' for c in cols)
    p = con.sql(f"SUMMARIZE SELECT {select} FROM {view}").df()
    profiles.append(p.assign(table=view))
profile = pd.concat(profiles, ignore_index=True)
profile["null_pct"] = pd.to_numeric(profile["null_percentage"], errors="coerce").round(2)
profile = profile[["table", "column_name", "column_type", "count", "null_pct", "approx_unique", "min", "max"]]
save(profile, "A2_missingness_profile.csv")
profile[(profile["null_pct"] > 0) | profile["column_name"].isin(KEY_COLUMNS)]

### A.3 How the joins expand measurement rows

The pipeline goes `wq_measurement` → `measurement_window` (each result × every linked dam that has a removal year) → pre/post only → pairs with `has_both` → `analysis_measurement`. **How to read it:** `rows_per_measurement` > 1 means the same measurement is reused for several dam exposures. That is by design (spec 3.11), not duplication, but it matters for inference (the same value enters several pairs).

In [ ]:
t0 = time.time()
expansion = au.counts_table(con, {
    "1. wq_measurement (cleaned results)": "SELECT measurement_id, station_id, series FROM m",
    "2. measurement_window, all periods": f"SELECT measurement_id, station_id, AR_ID, {SERIES} AS series FROM measurement_window",
    "3. measurement_window, pre/post only": f"SELECT measurement_id, station_id, AR_ID, {SERIES} AS series FROM measurement_window WHERE period IN ('pre', 'post')",
    "4. pre/post rows of has_both pair-series": f"""
        SELECT w.measurement_id, w.station_id, w.AR_ID, {au.series_sql('w.parameter_std', 'w.sample_fraction_std')} AS series
        FROM measurement_window w JOIN eligibility_pairs p
          ON w.AR_ID = p.AR_ID AND w.station_id = p.station_id AND w.parameter_std = p.parameter_std
         AND w.sample_fraction_std IS NOT DISTINCT FROM p.sample_fraction_std
        WHERE p.has_both AND w.period IN ('pre', 'post')""",
    "5. analysis_measurement": "SELECT measurement_id, station_id, AR_ID, series FROM am",
})
expansion["rows_per_measurement"] = (expansion["rows"] / expansion["measurements"]).round(2)
TIMINGS["join expansion"] = time.time() - t0
save(expansion, "A3_join_expansion.csv")
expansion

In [ ]:
# Where the measurement_window rows fall relative to each removal year.
q("""
    SELECT period, count(*) AS rows, count(DISTINCT measurement_id) AS measurements,
           count(DISTINCT AR_ID) AS dams, count(DISTINCT station_id) AS stations
    FROM measurement_window GROUP BY period ORDER BY rows DESC
""")

### A.4 Stations linked to several dams, and measurements with several exposures

**How to read it:** the first table bins stations by how many dams they link to: all links, links to dams with a removal year, and pairs that made it into `analysis_measurement`. The second bins each analysis measurement by how many dam exposures it is used for.

In [ ]:
BINS = ["1", "2", "3", "4", "5-9", "10+"]


def to_bin(n):
    return pd.cut(n, [0, 1, 2, 3, 4, 9, np.inf], labels=BINS)


per_station = q("""
    WITH links AS (
        SELECT l.station_id, count(*) AS n_all,
               count(*) FILTER (WHERE d.Year_Removed IS NOT NULL) AS n_year
        FROM dam_station_link l JOIN dam_removal d USING (AR_ID) GROUP BY l.station_id
    ),
    analysis AS (SELECT station_id, count(DISTINCT AR_ID) AS n_analysis FROM am GROUP BY station_id)
    SELECT * FROM links LEFT JOIN analysis USING (station_id)
""")
dams_per_station = pd.DataFrame({
    "all links": to_bin(per_station["n_all"]).value_counts(),
    "links to dams with a removal year": to_bin(per_station["n_year"].where(per_station["n_year"] > 0)).value_counts(),
    "pairs in analysis_measurement": to_bin(per_station["n_analysis"]).value_counts(),
}).reindex(BINS).fillna(0).astype(int).rename_axis("dams linked")
save(dams_per_station.reset_index(), "A4_dams_per_station.csv")
dams_per_station

In [ ]:
per_measurement = q("SELECT measurement_id, count(*) AS n FROM am GROUP BY measurement_id")
exposures = (per_measurement.assign(dam_exposures=to_bin(per_measurement["n"]))
             .groupby("dam_exposures", observed=False)
             .agg(measurements=("n", "size"), analysis_rows=("n", "sum")))
exposures["pct_measurements"] = (100 * exposures["measurements"] / exposures["measurements"].sum()).round(1)
exposures["pct_rows"] = (100 * exposures["analysis_rows"] / exposures["analysis_rows"].sum()).round(1)
save(exposures.reset_index(), "A4_measurement_exposures.csv")
exposures

### A.5 Genuine duplicates or reuse?

A repeated `measurement_id` in `analysis_measurement` is **reuse** if every copy has the same station, date, series and value and differs only in the dam (`AR_ID`). **How to read it:** `inconsistent_copies` should be 0 (then all repetition is reuse); `duplicate_keys` should be 0 (the primary key is `(measurement_id, AR_ID)`). Separate `measurement_id`s that look like the same result are a different question, covered in section C.

In [ ]:
reuse_check = q("""
    SELECT count(*) AS rows,
           count(DISTINCT measurement_id) AS measurements,
           count(*) - count(DISTINCT measurement_id) AS reused_rows,
           count(*) - count(DISTINCT CAST(measurement_id AS VARCHAR) || '|' || AR_ID) AS duplicate_keys,
           (SELECT count(*) FROM (
               SELECT measurement_id FROM am GROUP BY measurement_id
               HAVING count(DISTINCT station_id) > 1 OR count(DISTINCT sample_date) > 1
                   OR count(DISTINCT series) > 1 OR count(DISTINCT result_value_std) > 1
           )) AS inconsistent_copies
    FROM am
""")
save(reuse_check, "A5_reuse_check.csv")
reuse_check

### A.6 Spatial linkage

Snap distances (how far a point was moved to land on a stream reach), dam-to-network links and drainage-area availability. **How to read it:** large snap distances can mean a station was matched to the wrong reach; dams with no linked station can't enter the analysis; a drainage area of −9999 is an NHDPlus sentinel (often coastal or network-terminal reaches), not a real value.

In [ ]:
snap_quantiles = q("""
    WITH s AS (
        SELECT 'dams (all)' AS feature, snap_distance_m AS d FROM dam_removal
        UNION ALL SELECT 'dams (in analysis)', snap_distance_m FROM dam_removal WHERE AR_ID IN (SELECT AR_ID FROM pairs)
        UNION ALL SELECT 'stations (all)', snap_distance_m FROM wq_station
        UNION ALL SELECT 'stations (in analysis)', snap_distance_m FROM wq_station WHERE station_id IN (SELECT station_id FROM pairs)
    )
    SELECT feature, count(*) AS n, count(*) FILTER (WHERE d IS NULL) AS null_d,
           round(quantile_cont(d, 0.5), 1) AS median_m, round(quantile_cont(d, 0.9), 1) AS p90_m,
           round(quantile_cont(d, 0.99), 1) AS p99_m, round(max(d), 1) AS max_m
    FROM s GROUP BY feature ORDER BY feature
""")
over = q("SELECT 'dams (all)' AS feature, snap_distance_m AS d FROM dam_removal "
         "UNION ALL SELECT 'stations (all)', snap_distance_m FROM wq_station "
         "UNION ALL SELECT 'stations (in analysis)', snap_distance_m FROM wq_station "
         "WHERE station_id IN (SELECT station_id FROM pairs)")
for limit in THRESHOLDS["snap_distance_m"]:
    snap_quantiles[f"> {limit} m"] = snap_quantiles["feature"].map(
        over.assign(o=over["d"] > limit).groupby("feature")["o"].sum()).fillna(0).astype(int)
save(snap_quantiles, "A6_snap_distances.csv")
snap_quantiles

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
for feature in ["dams (all)", "stations (in analysis)"]:
    d = over.loc[over["feature"] == feature, "d"].dropna()
    ax.hist(np.log10(d.clip(lower=0.1)), bins=60, histtype="step", density=True, label=feature)
for limit in [100, 500, 5000]:
    ax.axvline(np.log10(limit), color="gray", ls="--", lw=0.8)
ax.set(xlabel="log10 snap distance (m)  [dashed: 100 m, 500 m, 5 km]", ylabel="density",
       title="Snap distances")
ax.legend()
plt.show()

In [ ]:
streams_check = ("(SELECT count(*) FROM dam_removal WHERE NHDPlusID NOT IN (SELECT nhdplus_id FROM streams))"
                 if "streams" in HAS else "NULL")
ws = "station_watershed" in HAS
linkage = q(f"""
    SELECT 'dams' AS item, 'total' AS check_name, count(*) AS n FROM dam_removal
    UNION ALL SELECT 'dams', 'no Year_Removed', count(*) FROM dam_removal WHERE Year_Removed IS NULL
    UNION ALL SELECT 'dams', 'NHDPlusID null', count(*) FROM dam_removal WHERE NHDPlusID IS NULL
    UNION ALL SELECT 'dams', 'NHDPlusID not in streams', {streams_check}
    UNION ALL SELECT 'dams', 'snap_fraction outside [0, 1]', count(*) FROM dam_removal WHERE snap_fraction NOT BETWEEN 0 AND 1
    UNION ALL SELECT 'dams', 'lat/lon missing, zero or out of range', count(*) FROM dam_removal
        WHERE Latitude IS NULL OR Longitude IS NULL OR Latitude = 0 OR Longitude = 0
           OR abs(Latitude) > 90 OR abs(Longitude) > 180
    UNION ALL SELECT 'dams', 'no linked station', count(*) FROM dam_removal WHERE AR_ID NOT IN (SELECT AR_ID FROM dam_station_link)
    UNION ALL SELECT 'dams', 'in analysis_measurement', count(DISTINCT AR_ID) FROM pairs
    UNION ALL SELECT 'stations', 'total', count(*) FROM wq_station
    UNION ALL SELECT 'stations', 'NHDPlusID null', count(*) FROM wq_station WHERE NHDPlusID IS NULL
    UNION ALL SELECT 'stations', 'lat/lon missing, zero or out of range', count(*) FROM wq_station
        WHERE latitude IS NULL OR longitude IS NULL OR latitude = 0 OR longitude = 0
           OR abs(latitude) > 90 OR abs(longitude) > 180
    UNION ALL SELECT 'stations', 'in analysis_measurement', count(DISTINCT station_id) FROM pairs
    UNION ALL SELECT 'links', 'total', count(*) FROM dam_station_link
    UNION ALL SELECT 'links', 'relationship = ' || relationship, count(*) FROM dam_station_link GROUP BY relationship
    UNION ALL SELECT 'links', 'distance_method = ' || coalesce(distance_method, '(null)'), count(*) FROM dam_station_link GROUP BY distance_method
    UNION ALL SELECT 'links', 'network_distance_km null', count(*) FROM dam_station_link WHERE network_distance_km IS NULL
    UNION ALL SELECT 'links', 'network_distance_km < 0', count(*) FROM dam_station_link WHERE network_distance_km < 0
    UNION ALL SELECT 'links', 'network_distance_km = 0', count(*) FROM dam_station_link WHERE network_distance_km = 0
    UNION ALL SELECT 'links', 'same_reach with tied snap_fraction (side null)', count(*)
        FROM dam_station_link l JOIN dam_removal d USING (AR_ID) JOIN wq_station s USING (station_id)
        WHERE l.relationship = 'same_reach' AND s.snap_fraction = d.snap_fraction
""")
if ws:
    linkage = pd.concat([linkage, q("""
        WITH a AS (SELECT DISTINCT station_id FROM pairs)
        SELECT 'analysis stations' AS item, 'no station_watershed row' AS check_name, count(*) AS n
            FROM a WHERE station_id NOT IN (SELECT station_id FROM station_watershed)
        UNION ALL SELECT 'analysis stations', 'nhdplus_total_area_km2 null', count(*)
            FROM a JOIN station_watershed USING (station_id) WHERE nhdplus_total_area_km2 IS NULL
        UNION ALL SELECT 'analysis stations', 'nhdplus_total_area_km2 <= 0 (e.g. -9999 sentinel)', count(*)
            FROM a JOIN station_watershed USING (station_id) WHERE nhdplus_total_area_km2 <= 0
        UNION ALL SELECT 'analysis stations', 'area_coverage > 1.05 (more area than NHDPlus)', count(*)
            FROM a JOIN station_watershed USING (station_id) WHERE area_coverage > 1.05
        UNION ALL SELECT 'analysis stations', 'area_coverage < 0.9', count(*)
            FROM a JOIN station_watershed USING (station_id) WHERE area_coverage < 0.9
        UNION ALL SELECT 'analysis stations', 'truncated watershed', count(*)
            FROM a JOIN station_watershed USING (station_id) WHERE truncated
    """)], ignore_index=True)
save(linkage, "A6_spatial_linkage_checks.csv")
linkage

## B. Measurement quality and units

### B.1 Parameters, fractions, units and providers

One row per (standard parameter, WQP name, reported fraction, standard fraction, reported unit, provider). **How to read it:** the standard unit is the same within a `parameter_std`; the reported unit shows what was converted. `pct_unit_basis_assumed` is the share of nutrient values reported as bare `mg/L` whose N/P basis was assumed (spec 3.6.4). Fractions other than `Dissolved`/`Total` (e.g. `Suspended`, `Inorganic`) and null fractions on nutrients are separate series downstream.

In [ ]:
puf = q("""
    SELECT parameter_std, parameter, coalesce(sample_fraction, '(blank)') AS sample_fraction,
           coalesce(sample_fraction_std, '(null)') AS sample_fraction_std,
           coalesce(result_unit_raw, '(blank)') AS result_unit_raw, result_unit_std, provider,
           count(*) AS rows, count(DISTINCT station_id) AS stations,
           round(100 * avg(is_censored::INT), 1) AS pct_censored,
           round(100 * avg(unit_basis_assumed::INT), 1) AS pct_unit_basis_assumed
    FROM m GROUP BY ALL ORDER BY parameter_std, rows DESC
""")
save(puf, "B1_parameter_unit_fraction_provider.csv")
print(f"{len(puf)} combinations (all saved); largest 60 shown")
puf.sort_values("rows", ascending=False).head(60)

### B.2 Missing or ambiguous units

A blank reported unit means the unit was taken from the detection limit, the USGS parameter code, or the pH default (spec 3.6.4). **How to read it:** high `pct_blank_unit` or `pct_unit_basis_assumed` means the standard unit rests on an inference for that share of rows; turbidity's NTU / FNU / NTRU are different instruments treated as comparable.

In [ ]:
display(q("""
    SELECT parameter_std, count(*) AS rows,
           round(100 * avg((result_unit_raw IS NULL)::INT), 2) AS pct_blank_unit,
           round(100 * avg(unit_basis_assumed::INT), 2) AS pct_unit_basis_assumed,
           round(100 * avg(unit_basis_assumed::INT) FILTER (WHERE provider = 'NWIS'), 2) AS pct_assumed_nwis,
           round(100 * avg(unit_basis_assumed::INT) FILTER (WHERE provider = 'STORET'), 2) AS pct_assumed_storet
    FROM m GROUP BY parameter_std ORDER BY rows DESC
"""))
print("Organizations contributing the most unit-basis-assumed rows:")
display(q("""
    SELECT organization_id, parameter_std, count(*) AS assumed_rows
    FROM m WHERE unit_basis_assumed GROUP BY ALL ORDER BY assumed_rows DESC LIMIT 15
"""))
print("Turbidity units by provider:")
q("""
    SELECT coalesce(result_unit_raw, '(blank)') AS unit, provider, count(*) AS rows,
           count(DISTINCT station_id) AS stations
    FROM m WHERE parameter_std = 'turbidity' GROUP BY ALL ORDER BY rows DESC
""")

### B.3 Analytical methods and speciation

`wq_measurement` doesn't keep the method fields, but `wq_measurement_raw` (every downloaded row, verbatim) does, keyed by the same `measurement_id`. This reads only the method columns for the cleaned measurements (raw rows repeated across batches are identical, so any copy will do). **How to read it:** many distinct methods within a series is a heterogeneity risk; `MethodSpeciationName` shows the stated chemical form of nutrient results (it resolves "bare mg/L" where present).

In [ ]:
METHOD_FIELDS = {
    "method_id": "ResultAnalyticalMethod/MethodIdentifier",
    "method_context": "ResultAnalyticalMethod/MethodIdentifierContext",
    "method_name": "ResultAnalyticalMethod/MethodName",
    "speciation": "MethodSpeciationName",
}
HAS_METHODS = False
if RUN_METHOD_AUDIT and "wq_measurement_raw" in HAS:
    raw_columns = set(con.sql("SELECT * FROM wq_measurement_raw LIMIT 0").columns)
    present = {k: v for k, v in METHOD_FIELDS.items() if v in raw_columns}
    absent = sorted(set(METHOD_FIELDS) - set(present))
    print("Method fields found:", list(present), "| not in wq_measurement_raw:", absent or "none")
    if present:
        t0 = time.time()
        select = ", ".join(f'any_value("{v}") AS {k}' for k, v in present.items())
        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE m_method AS
            SELECT CAST(measurement_id AS BIGINT) AS measurement_id, {select}
            FROM wq_measurement_raw
            WHERE CAST(measurement_id AS BIGINT) IN (SELECT measurement_id FROM m)
            GROUP BY 1
        """)
        for k in METHOD_FIELDS:
            if k not in present:
                con.execute(f"ALTER TABLE m_method ADD COLUMN {k} VARCHAR")
        HAS_METHODS = True
        TIMINGS["method fields"] = time.time() - t0
        print(f"Method fields for {con.sql('SELECT count(*) FROM m_method').fetchone()[0]:,} measurements "
              f"({TIMINGS['method fields']:.0f} s)")
else:
    print("Skipped (RUN_METHOD_AUDIT is False or wq_measurement_raw is missing).")

In [ ]:
if HAS_METHODS:
    methods = q("""
        SELECT m.series, coalesce(x.method_context, '') || ':' || coalesce(x.method_id, '(none)') AS method,
               count(*) AS rows, count(DISTINCT m.station_id) AS stations
        FROM m LEFT JOIN m_method x USING (measurement_id) GROUP BY ALL
    """)
    by_series = methods.groupby("series").agg(rows=("rows", "sum"), distinct_methods=("method", "nunique"))
    no_method = methods[methods["method"].str.endswith("(none)")].groupby("series")["rows"].sum()
    by_series["pct_no_method"] = (100 * no_method.reindex(by_series.index).fillna(0) / by_series["rows"]).round(1)
    top = (methods.sort_values("rows", ascending=False).groupby("series").head(3)
           .assign(pct=lambda d: (100 * d["rows"] / d["series"].map(by_series["rows"])).round(1)))
    by_series["top_methods"] = top.groupby("series").apply(
        lambda d: "; ".join(f"{m} ({p}%)" for m, p in zip(d["method"], d["pct"])))
    save(methods, "B3_methods_by_series.csv")
    display(by_series.sort_values("rows", ascending=False))
    print("Stated speciation of nutrient results (bare mg/L rows are the ones it can resolve):")
    display(q("""
        SELECT m.parameter_std, m.unit_basis_assumed, coalesce(x.speciation, '(blank)') AS speciation, count(*) AS rows
        FROM m JOIN m_method x USING (measurement_id)
        WHERE m.parameter_std IN ('nitrate', 'ammonia', 'orthophosphate', 'phosphorus')
        GROUP BY ALL ORDER BY parameter_std, rows DESC
    """))

### B.4 Exact zeros, boundary values and extremes

The cleaning step only drops values outside wide physical limits (`wq_rules.PLAUSIBLE_RANGES`). These screens look for values that pass those limits but are still suspect:

- **zeros**: implausible for pH, conductance and TDS; rare for DO, DO saturation and turbidity; for nutrients a reported 0 (not flagged as censored) usually means "below detection".
- **at a plausible bound**: values sitting exactly on a `PLAUSIBLE_RANGES` limit (e.g. DO = 25, DO saturation = 300) often come from instrument caps or entry codes.
- **beyond a suspicious limit**: tighter, exploratory limits in `SUSPICIOUS_LIMITS` (configuration cell).

All of these are **flags**; nothing is removed.

In [ ]:
limits = pd.DataFrame([{
    "parameter_std": p,
    "plausible_low": wq_rules.PLAUSIBLE_RANGES[p][0], "plausible_high": wq_rules.PLAUSIBLE_RANGES[p][1],
    "suspicious_low": SUSPICIOUS_LIMITS.get(p, (None, None))[0],
    "suspicious_high": SUSPICIOUS_LIMITS.get(p, (None, None))[1],
    "zero_rule": ZERO_RULES.get(p, "plausible"),
} for p in wq_rules.PLAUSIBLE_RANGES])
for c in ["suspicious_low", "suspicious_high"]:
    limits[c] = pd.to_numeric(limits[c]).astype("float64")
con.register("limits_df", limits)
con.execute("CREATE OR REPLACE TEMP TABLE limits AS SELECT * FROM limits_df")

value_screens = q("""
    SELECT m.parameter_std, any_value(l.zero_rule) AS zero_rule, count(*) AS rows,
           count(*) FILTER (WHERE result_value_std = 0) AS zeros,
           count(*) FILTER (WHERE result_value_std = 0 AND NOT is_censored) AS zeros_not_censored,
           count(*) FILTER (WHERE result_value_std < 0) AS negatives,
           count(*) FILTER (WHERE result_value_std = l.plausible_low AND l.plausible_low <> 0) AS at_low_bound,
           count(*) FILTER (WHERE result_value_std = l.plausible_high) AS at_high_bound,
           count(*) FILTER (WHERE result_value_std < l.suspicious_low) AS below_suspicious,
           count(*) FILTER (WHERE result_value_std > l.suspicious_high) AS above_suspicious,
           quantile_cont(result_value_std, 0.001) AS p0_1, quantile_cont(result_value_std, 0.01) AS p1,
           quantile_cont(result_value_std, 0.5) AS p50, quantile_cont(result_value_std, 0.99) AS p99,
           quantile_cont(result_value_std, 0.999) AS p99_9, max(result_value_std) AS max
    FROM m JOIN limits l USING (parameter_std)
    GROUP BY m.parameter_std ORDER BY rows DESC
""")
save(value_screens, "B4_value_screens.csv")
value_screens.round(4)

In [ ]:
# Row-level value flags (one row per measurement and reason), for examples and section I.
con.execute("""
    CREATE OR REPLACE TEMP TABLE value_flags AS
    SELECT measurement_id, parameter_std, reason FROM (
        SELECT m.measurement_id, m.parameter_std,
               CASE
                   WHEN m.result_value_std = 0 AND l.zero_rule = 'implausible' THEN 'zero (implausible)'
                   WHEN m.result_value_std = 0 AND l.zero_rule = 'should_be_censored' AND NOT m.is_censored
                       THEN 'zero (nutrient not flagged censored)'
                   WHEN m.result_value_std = 0 AND l.zero_rule = 'rare' THEN 'zero (rare)'
               END AS reason
        FROM m JOIN limits l USING (parameter_std)
        UNION ALL
        SELECT m.measurement_id, m.parameter_std, 'at plausible bound'
        FROM m JOIN limits l USING (parameter_std)
        WHERE m.result_value_std = l.plausible_high OR (m.result_value_std = l.plausible_low AND l.plausible_low <> 0)
        UNION ALL
        SELECT m.measurement_id, m.parameter_std, 'beyond suspicious limit'
        FROM m JOIN limits l USING (parameter_std)
        WHERE m.result_value_std < l.suspicious_low OR m.result_value_std > l.suspicious_high
    ) WHERE reason IS NOT NULL
""")
value_flag_summary = q("""
    SELECT f.parameter_std, f.reason, count(*) AS rows, count(DISTINCT m.station_id) AS stations,
           round(100 * avg((m.provider = 'NWIS')::INT), 1) AS pct_nwis
    FROM value_flags f JOIN m USING (measurement_id) GROUP BY ALL ORDER BY rows DESC
""")
save(value_flag_summary, "B4_value_flag_summary.csv")
display(value_flag_summary)
print("Examples (3 per parameter and reason):")
examples = q("""
    SELECT f.parameter_std, f.reason, m.station_id, m.organization_id, m.sample_date, m.result_value_raw,
           m.result_unit_raw, m.result_value_std, m.is_censored, m.detection_condition
    FROM value_flags f JOIN m USING (measurement_id)
    QUALIFY row_number() OVER (PARTITION BY f.parameter_std, f.reason ORDER BY hash(m.measurement_id)) <= 3
    ORDER BY f.parameter_std, f.reason
""")
save(examples, "B4_value_flag_examples.csv")
examples

### B.5 Possible unit errors within stations

Two screens, both within one station and series:

1. **Blocks**: rows grouped by organization, reported unit and assumed-basis flag. If a block's median differs from the station's main block by close to a known factor (×1000 for mg vs µg or µS vs mS, within 50%; ×4.43 NO₃ vs N, ×3.07 PO₄ vs P, ×1.29 NH₄ vs N, within 15%), it is a **candidate** unit or basis error. Source metadata (unit, organization, speciation) is shown so a correction could be justified, but none is applied.
2. **Single values** about 1000× (or 1/1000×) the station median (within 50%).

pH, temperature, DO saturation and turbidity are left out: they have no mass-unit slip, and storm turbidity can genuinely be 1000× a station's median.

**How to read it:** a candidate is a lead to check, not proof. A real step change (e.g. a new treatment plant) can produce a ratio near 3 by coincidence.

In [ ]:
tol_mass = THRESHOLDS["unit_factor_tolerance"]["x1000"]
tol_basis = THRESHOLDS["unit_factor_tolerance"]["basis"]
tolerances = {label: tol_mass if "1000" in label else tol_basis for label in au.UNIT_FACTORS}
# Parameters where a x1000 or basis slip can't happen, or 1000x swings are natural (storm turbidity).
NO_UNIT_SCREEN = ["ph", "water_temp", "do_saturation", "turbidity"]
blocks = q(f"""
    WITH b AS (
        SELECT station_id, series, parameter_std, organization_id, provider,
               coalesce(result_unit_raw, '(blank)') AS unit_raw, unit_basis_assumed,
               count(*) AS n, median(result_value_std) AS block_median,
               min(sample_date) AS first_date, max(sample_date) AS last_date
        FROM m WHERE NOT is_censored AND result_value_std > 0
        GROUP BY ALL HAVING count(*) >= {THRESHOLDS['unit_block_min_n']}
    ),
    ref AS (
        SELECT station_id, series, arg_max(block_median, n) AS ref_median, max(n) AS ref_n, count(*) AS n_blocks
        FROM b GROUP BY ALL
    )
    SELECT b.*, ref.ref_median, ref.ref_n, ref.n_blocks, b.block_median / ref.ref_median AS ratio
    FROM b JOIN ref USING (station_id, series)
    WHERE ref.n_blocks > 1 AND b.n < ref.ref_n
""")
# Which factors make sense for which parameter.
ALLOWED = {
    "nitrate": ["x1000 (mg vs ug)", "/1000 (ug vs mg)", "NO3 vs N (x4.43)", "N vs NO3 (/4.43)"],
    "ammonia": ["x1000 (mg vs ug)", "/1000 (ug vs mg)", "NH4 vs N (x1.29)", "N vs NH4 (/1.29)"],
    "orthophosphate": ["x1000 (mg vs ug)", "/1000 (ug vs mg)", "PO4 vs P (x3.07)", "P vs PO4 (/3.07)"],
    "phosphorus": ["x1000 (mg vs ug)", "/1000 (ug vs mg)", "PO4 vs P (x3.07)", "P vs PO4 (/3.07)"],
}
default_factors = ["x1000 (mg vs ug)", "/1000 (ug vs mg)"]
blocks["candidate_factor"] = [
    au.nearest_factor(r, tolerances, {k: au.UNIT_FACTORS[k] for k in ALLOWED.get(p, default_factors)})
    if p not in NO_UNIT_SCREEN else None
    for r, p in zip(blocks["ratio"], blocks["parameter_std"])
]
unit_candidates = blocks[blocks["candidate_factor"].notna()].sort_values("n", ascending=False)
save(unit_candidates, "B5_unit_block_candidates.csv")
print(f"{len(blocks):,} minority blocks compared; {len(unit_candidates):,} near a unit factor "
      f"({unit_candidates['n'].sum():,} rows)")
display(unit_candidates.groupby(["parameter_std", "candidate_factor"])
        .agg(blocks=("n", "size"), rows=("n", "sum"), stations=("station_id", "nunique")))
unit_candidates.head(15)

In [ ]:
band = np.log10(1 + tol_mass)
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE unit_value_flags AS
    WITH s AS (
        SELECT station_id, series, median(result_value_std) AS med
        FROM m WHERE NOT is_censored AND result_value_std > 0
        GROUP BY ALL HAVING count(*) >= {THRESHOLDS['mad_min_n']}
    )
    SELECT m.measurement_id, m.parameter_std, m.station_id,
           CASE WHEN log10(m.result_value_std / s.med) > 0 THEN 'about x1000 station median'
                ELSE 'about /1000 station median' END AS reason
    FROM m JOIN s USING (station_id, series)
    WHERE m.result_value_std > 0 AND NOT m.is_censored AND m.parameter_std NOT IN ({sql_list(NO_UNIT_SCREEN)})
      AND abs(abs(log10(m.result_value_std / s.med)) - 3) <= {band}
""")
unit_value_summary = q("SELECT parameter_std, reason, count(*) AS rows, count(DISTINCT station_id) AS stations "
                       "FROM unit_value_flags GROUP BY ALL ORDER BY rows DESC")
save(unit_value_summary, "B5_unit_value_flags_summary.csv")
unit_value_summary

### B.6 Consistency between related measurements

Compares values that should agree when taken in the **same sampling event**. Only uncensored values are compared (a censored value is half the detection limit, not a measurement), and only where each series has exactly one value in the event, so depth profiles don't pair the wrong rows. Two matching bases are reported separately:

- `same_activity`: same station and WQP `ActivityIdentifier`, the strictest definition of one sample.
- `same_station_day`: same station and date. Broader; STORET often files field readings and lab results under different activities.

Checks:
- a part exceeding its whole: dissolved > total (P, nitrate, ammonia); orthophosphate > total phosphorus;
- TDS / specific conductance outside the configured range (typically 0.55–0.75);
- reported DO saturation vs saturation computed from DO and water temperature (fresh water, 1 atm; high-elevation sites read higher than computed, so a small positive bias is expected).

**How to read it:** `pct_violations` is the share of comparable events that disagree beyond the tolerances. These are **flags** on both measurements of a disagreeing event.

In [ ]:
rel = THRESHOLDS["consistency_rel_tol"]
abs_tol = THRESHOLDS["consistency_abs_tol"]
lo, hi = THRESHOLDS["tds_sc_ratio"]
CHECK_SERIES = ["phosphorus (Dissolved)", "phosphorus (Total)", "orthophosphate (Dissolved)",
                "orthophosphate (Total)", "nitrate (Dissolved)", "nitrate (Total)",
                "ammonia (Dissolved)", "ammonia (Total)", "tds", "specific_conductance",
                "dissolved_oxygen", "do_saturation", "water_temp"]
base = f"""
    SELECT measurement_id, station_id, activity_id, sample_date, series, provider, result_value_std AS v
    FROM m WHERE NOT is_censored AND result_value_std IS NOT NULL AND series IN ({sql_list(CHECK_SERIES)})
"""
con.execute(f"CREATE OR REPLACE TEMP TABLE ev_activity AS SELECT * FROM ({base}) WHERE activity_id IS NOT NULL "
            "QUALIFY count(*) OVER (PARTITION BY station_id, activity_id, series) = 1")
con.execute(f"CREATE OR REPLACE TEMP TABLE ev_day AS SELECT * FROM ({base}) "
            "QUALIFY count(*) OVER (PARTITION BY station_id, sample_date, series) = 1")

PART_WHOLE = [
    ("dissolved P > total P", "phosphorus (Dissolved)", "phosphorus (Total)", abs_tol["phosphorus"]),
    ("dissolved ortho-P > total P", "orthophosphate (Dissolved)", "phosphorus (Total)", abs_tol["orthophosphate"]),
    ("total ortho-P > total P", "orthophosphate (Total)", "phosphorus (Total)", abs_tol["orthophosphate"]),
    ("dissolved ortho-P > dissolved P", "orthophosphate (Dissolved)", "phosphorus (Dissolved)", abs_tol["orthophosphate"]),
    ("dissolved nitrate > total nitrate", "nitrate (Dissolved)", "nitrate (Total)", abs_tol["nitrate"]),
    ("dissolved ammonia > total ammonia", "ammonia (Dissolved)", "ammonia (Total)", abs_tol["ammonia"]),
]
parts = []
for basis, table, key in [("same_activity", "ev_activity", "activity_id"), ("same_station_day", "ev_day", "sample_date")]:
    for name, part, whole, a_tol in PART_WHOLE:
        parts.append(f"""
            SELECT '{name}' AS check_name, '{basis}' AS basis, a.station_id, a.sample_date, a.provider,
                   a.measurement_id AS id_a, b.measurement_id AS id_b, NULL::BIGINT AS id_c,
                   a.v AS value_a, b.v AS value_b, a.v / nullif(b.v, 0) AS metric,
                   a.v > b.v * (1 + {rel}) AND a.v - b.v > {a_tol} AS violation
            FROM {table} a JOIN {table} b ON a.station_id = b.station_id AND a.{key} = b.{key}
            WHERE a.series = '{part}' AND b.series = '{whole}'""")
    parts.append(f"""
        SELECT 'TDS / conductance outside [{lo}, {hi}]', '{basis}', a.station_id, a.sample_date, a.provider,
               a.measurement_id, b.measurement_id, NULL::BIGINT, a.v, b.v, a.v / nullif(b.v, 0),
               b.v > 0 AND (a.v / b.v < {lo} OR a.v / b.v > {hi})
        FROM {table} a JOIN {table} b ON a.station_id = b.station_id AND a.{key} = b.{key}
        WHERE a.series = 'tds' AND b.series = 'specific_conductance'""")
    parts.append(f"""
        SELECT 'DO saturation: reported vs computed', '{basis}', a.station_id, a.sample_date, a.provider,
               a.measurement_id, s.measurement_id, t.measurement_id, s.v,
               100 * a.v / {au.do_saturation_sql('t.v')}, s.v - 100 * a.v / {au.do_saturation_sql('t.v')},
               abs(s.v - 100 * a.v / {au.do_saturation_sql('t.v')}) > {THRESHOLDS['do_saturation_tol_pct']}
        FROM {table} a
        JOIN {table} t ON t.station_id = a.station_id AND t.{key} = a.{key} AND t.series = 'water_temp'
        JOIN {table} s ON s.station_id = a.station_id AND s.{key} = a.{key} AND s.series = 'do_saturation'
        WHERE a.series = 'dissolved_oxygen'""")
t0 = time.time()
con.execute("CREATE OR REPLACE TEMP TABLE consistency AS " + " UNION ALL ".join(parts))
TIMINGS["consistency checks"] = time.time() - t0
consistency_summary = q("""
    SELECT check_name, basis, count(*) AS comparisons, count(*) FILTER (WHERE violation) AS violations,
           round(100 * avg(violation::INT), 2) AS pct_violations, round(median(metric), 3) AS median_metric,
           count(DISTINCT station_id) FILTER (WHERE violation) AS stations_with_violations
    FROM consistency GROUP BY ALL ORDER BY check_name, basis
""")
save(consistency_summary, "B6_consistency_summary.csv")
print("metric = part / whole for part-whole checks; TDS / SC for the ratio check; "
      "reported - computed (percentage points) for DO saturation")
consistency_summary

In [ ]:
consistency_examples = q("""
    SELECT * FROM consistency WHERE violation
    QUALIFY row_number() OVER (PARTITION BY check_name, basis ORDER BY hash(id_a)) <= 3
    ORDER BY check_name, basis
""")
con.execute("""
    CREATE OR REPLACE TEMP TABLE consistency_flags AS
    SELECT DISTINCT measurement_id, check_name FROM (
        SELECT id_a AS measurement_id, check_name FROM consistency WHERE violation
        UNION ALL SELECT id_b, check_name FROM consistency WHERE violation
        UNION ALL SELECT id_c, check_name FROM consistency WHERE violation AND id_c IS NOT NULL
    )
""")
save(q("SELECT * FROM consistency WHERE violation"), "B6_consistency_violations.parquet")
consistency_examples

In [ ]:
# Distribution of the TDS / conductance ratio and of the DO saturation difference (same station-day).
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
ratio = q("SELECT metric FROM consistency WHERE check_name LIKE 'TDS%' AND basis = 'same_station_day'")["metric"]
axes[0].hist(ratio.clip(0, 2), bins=80)
for x in THRESHOLDS["tds_sc_ratio"]:
    axes[0].axvline(x, color="k", ls="--", lw=1)
axes[0].set(title="TDS / specific conductance (clipped at 2)", xlabel="ratio")
diff = q("SELECT metric FROM consistency WHERE check_name LIKE 'DO%' AND basis = 'same_station_day'")["metric"]
axes[1].hist(diff.clip(-60, 60), bins=80)
axes[1].axvline(0, color="k", lw=1)
axes[1].set(title="DO saturation: reported − computed", xlabel="percentage points (clipped ±60)")
fig.tight_layout()
plt.show()

## C. Duplicate and repeated-measurement diagnostics

The cleaning step removed exact repeats (same `measurement_id`) and the same result filed under two activities with the same time and value (spec 3.6.5). This section looks for what could remain, **without assuming that the same station, date and parameter means a duplicate**: several readings on one day are often real (depth profiles, repeated field readings).

### C.1 Possible duplicate records

| check | meaning |
|---|---|
| (a) | identical reported fields (org, activity, time, name, fraction, raw value, raw unit, detection condition) but different `measurement_id`: the hash differs only by the method ID, i.e. the same result reported under two methods |
| (b) | same station, date, time, series and value, different activity: should be ~0 after dedup 2 |
| (c) | same station, date, series and value, but different (or missing) times |
| (d) | like (c), but the copies come from different providers (NWIS vs STORET) |

In [ ]:
dup_checks = q("""
    WITH a AS (SELECT count(*) AS n FROM m GROUP BY station_id, organization_id, activity_id, sample_date,
                   sample_time, parameter, sample_fraction, result_value_raw, result_unit_raw, detection_condition
               HAVING count(*) > 1),
    b AS (SELECT count(*) AS n FROM m GROUP BY station_id, sample_date, sample_time, series, result_value_std
          HAVING count(*) > 1 AND count(DISTINCT activity_id) > 1),
    c AS (SELECT count(*) AS n FROM m GROUP BY station_id, sample_date, series, result_value_std
          HAVING count(*) > 1 AND (count(DISTINCT sample_time) > 1 OR count(sample_time) < count(*))),
    d AS (SELECT count(*) AS n FROM m GROUP BY station_id, sample_date, series, result_value_std
          HAVING count(DISTINCT provider) > 1)
    SELECT '(a) same reported fields, different measurement_id' AS check_name, count(*) AS groups, sum(n) AS rows, sum(n) - count(*) AS extra_rows FROM a
    UNION ALL SELECT '(b) same station/date/time/series/value, different activity', count(*), sum(n), sum(n) - count(*) FROM b
    UNION ALL SELECT '(c) same station/date/series/value, different or missing time', count(*), sum(n), sum(n) - count(*) FROM c
    UNION ALL SELECT '(d) same station/date/series/value, different providers', count(*), sum(n), sum(n) - count(*) FROM d
""")
save(dup_checks, "C1_duplicate_checks.csv")
dup_checks

In [ ]:
# (c) by series: where same-value, same-day repeats concentrate.
q("""
    SELECT series, count(*) AS groups, sum(n) AS rows
    FROM (SELECT series, count(*) AS n FROM m GROUP BY station_id, sample_date, series, result_value_std
          HAVING count(*) > 1 AND (count(DISTINCT sample_time) > 1 OR count(sample_time) < count(*)))
    GROUP BY series ORDER BY rows DESC
""")

### C.2 Co-located stations reporting the same values

Different agencies sometimes report the same sample under their own station IDs (e.g. a state copying USGS results). Candidates: two stations snapped to the **same NHDPlus reach** and within `colocated_max_m` of each other (original coordinates), with the same series on the same date and values within `duplicate_rel_tol`. **How to read it:** two nearby stations sampling the same river on the same day can legitimately agree, so exact agreement across many dates (`exact_matches` close to `matches`) is the strongest sign of a re-reported record.

In [ ]:
t0 = time.time()
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE colocated AS
    SELECT a.station_id AS station_a, b.station_id AS station_b,
           a.organization_id AS org_a, b.organization_id AS org_b, a.NHDPlusID AS reach,
           {au.haversine_m_sql('a.latitude', 'a.longitude', 'b.latitude', 'b.longitude')} AS distance_m
    FROM wq_station a JOIN wq_station b ON a.NHDPlusID = b.NHDPlusID AND a.station_id < b.station_id
    WHERE {au.haversine_m_sql('a.latitude', 'a.longitude', 'b.latitude', 'b.longitude')} <= {THRESHOLDS['colocated_max_m']}
""")
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE colocated_matches AS
    SELECT c.*, x.series, x.sample_date, x.provider AS provider_a, y.provider AS provider_b,
           x.measurement_id AS id_a, y.measurement_id AS id_b,
           x.result_value_std AS value_a, y.result_value_std AS value_b,
           x.result_value_std = y.result_value_std AS exact
    FROM colocated c
    JOIN m x ON x.station_id = c.station_a
    JOIN m y ON y.station_id = c.station_b AND y.sample_date = x.sample_date AND y.series = x.series
    WHERE abs(x.result_value_std - y.result_value_std)
          <= greatest({THRESHOLDS['duplicate_rel_tol']} * abs(x.result_value_std), 1e-9)
""")
TIMINGS["co-located stations"] = time.time() - t0
colocated_summary = q("""
    SELECT (SELECT count(*) FROM colocated) AS colocated_station_pairs,
           (SELECT count(*) FROM colocated WHERE org_a <> org_b) AS pairs_different_orgs,
           count(DISTINCT station_a || station_b) AS station_pairs_with_matches,
           count(*) AS matches, count(*) FILTER (WHERE exact) AS exact_matches,
           count(*) FILTER (WHERE provider_a <> provider_b) AS cross_provider_matches
    FROM colocated_matches
""")
save(colocated_summary, "C2_colocated_summary.csv")
display(colocated_summary)
by_pair = q("""
    SELECT station_a, station_b, org_a, org_b, round(any_value(distance_m)) AS distance_m,
           count(*) AS matches, count(*) FILTER (WHERE exact) AS exact,
           count(DISTINCT sample_date) AS dates, string_agg(DISTINCT series, ', ') AS series
    FROM colocated_matches GROUP BY ALL ORDER BY matches DESC
""")
save(by_pair, "C2_colocated_station_pairs.csv")
save(q("SELECT * FROM colocated_matches"), "C2_colocated_matches.parquet")
by_pair.head(20)

### C.3 Several readings on the same date

Groups of station × series × date with more than one row, classified by what differs:

- `depth profile`: more than one sample depth;
- `different times`: one depth (or none), several sample times;
- `same time/no time, different values`: e.g. repeated field readings;
- `identical values`: same value repeated.

The last columns are a **hypothetical** aggregation, counted only: rows if the data were collapsed to one value per station × series × day, or per station × series × day × depth (keeping profiles).

In [ ]:
t0 = time.time()
con.execute("""
    CREATE OR REPLACE TEMP TABLE same_day AS
    SELECT station_id, series, sample_date, count(*) AS n,
           count(DISTINCT sample_time) AS n_times, count(DISTINCT depth_m) AS n_depths,
           count(DISTINCT result_value_std) AS n_values, count(DISTINCT provider) AS n_providers,
           count(DISTINCT coalesce(CAST(depth_m AS VARCHAR), '-')) AS n_depth_groups
    FROM m GROUP BY station_id, series, sample_date
""")
same_day = q("""
    SELECT series, sum(n) AS rows, count(*) AS station_days,
           count(*) FILTER (WHERE n > 1) AS multi_reading_days,
           sum(n) FILTER (WHERE n > 1) AS rows_on_multi_days,
           count(*) FILTER (WHERE n > 1 AND n_depths > 1) AS depth_profile,
           count(*) FILTER (WHERE n > 1 AND n_depths <= 1 AND n_times > 1) AS different_times,
           count(*) FILTER (WHERE n > 1 AND n_depths <= 1 AND n_times <= 1 AND n_values > 1) AS same_time_different_values,
           count(*) FILTER (WHERE n > 1 AND n_values = 1) AS identical_values,
           max(n) AS max_per_day,
           count(*) AS hypothetical_rows_daily,
           sum(n_depth_groups) AS hypothetical_rows_daily_by_depth
    FROM same_day GROUP BY series ORDER BY rows DESC
""")
same_day["pct_rows_on_multi_days"] = (100 * same_day["rows_on_multi_days"].fillna(0) / same_day["rows"]).round(1)
TIMINGS["same-day readings"] = time.time() - t0
save(same_day, "C3_same_day_readings.csv")
same_day

### C.4 Repeated measurements per station and series

**How to read it:** most station-series have a handful to a few hundred values over decades; a very high `max_per_day` or `p99` points to high-frequency (sensor-like) records that would dominate a row-weighted model.

In [ ]:
per_station_series = q("""
    WITH s AS (SELECT station_id, series, count(*) AS n, count(DISTINCT year(sample_date)) AS years
               FROM m GROUP BY ALL)
    SELECT series, count(*) AS station_series, round(median(n)) AS median_rows,
           round(quantile_cont(n, 0.9)) AS p90_rows, round(quantile_cont(n, 0.99)) AS p99_rows, max(n) AS max_rows,
           round(median(n / years), 1) AS median_rows_per_year
    FROM s GROUP BY series ORDER BY station_series DESC
""")
save(per_station_series, "C4_rows_per_station_series.csv")
display(per_station_series)
print("Busiest station-series (most rows):")
q("""
    SELECT m.station_id, m.series, count(*) AS rows, count(DISTINCT sample_date) AS dates,
           max(sd.n) AS max_per_day, any_value(m.organization_id) AS organization_id
    FROM m JOIN same_day sd USING (station_id, series, sample_date)
    GROUP BY ALL ORDER BY rows DESC LIMIT 10
""")

### C.5 Reuse of one measurement across several dam-station links

Per series in `analysis_measurement`: `reuse_rows` are rows that repeat a measurement already used for another dam. **How to read it:** a high `pct_reuse` means the effective sample size is much smaller than the row count suggests.

In [ ]:
reuse = q("""
    SELECT series, count(*) AS rows, count(DISTINCT measurement_id) AS measurements,
           count(*) - count(DISTINCT measurement_id) AS reuse_rows,
           round(100 * (count(*) - count(DISTINCT measurement_id)) / count(*), 1) AS pct_reuse,
           count(DISTINCT pair) AS pairs, count(DISTINCT station_id) AS stations
    FROM am GROUP BY series ORDER BY rows DESC
""")
save(reuse, "C5_reuse_by_series.csv")
reuse

## D. Robust outlier flags

Within each **station × series**, a robust z-score:

`z = (x − median) / (1.4826 × MAD)`

where MAD is the median absolute deviation, so `|z|` is comparable to a standard z-score for normal data but isn't pulled by the outliers themselves. Details:

- **Scale:** positive, right-skewed parameters (`LOG_PARAMETERS`) are screened on log10; the others on their own scale. Both scales are computed for the log parameters so they can be compared.
- **Reference values:** the median and MAD come from **uncensored** values (censored ones are half a detection limit), but every row is scored.
- **Zeros and negatives** can't be logged: they are `not_evaluable` on the log scale (they are covered by the value screens in B.4).
- **Sparse groups** (fewer than `mad_min_n` reference values) are not screened.
- **Zero MAD** (over half the values identical): falls back to the mean absolute deviation (`z = (x − median) / (1.2533 × MeanAD)`); if every value is identical, the group is `zero_spread` and no z is computed.
- **Threshold:** `|z| > mad_threshold` (5 to start); section D.3 shows other thresholds.

Flags are saved to a new file; **no observation is removed**.

In [ ]:
t0 = time.time()
source = f"""
    SELECT measurement_id, station_id, series, parameter_std, provider, is_censored, month, sample_date,
           result_value_std AS value, 'log10' AS scale,
           CASE WHEN result_value_std > 0 THEN log10(result_value_std) END AS x
    FROM m WHERE parameter_std IN ({LOG_LIST})
    UNION ALL
    SELECT measurement_id, station_id, series, parameter_std, provider, is_censored, month, sample_date,
           result_value_std, 'linear', result_value_std
    FROM m
"""
au.robust_z(con, source, ["station_id", "series", "scale"], "robust_all",
            THRESHOLDS["mad_min_n"], reference_filter="NOT is_censored")
T = THRESHOLDS["mad_threshold"]
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE robust AS
    SELECT *, abs(robust_z) > {T} AS flagged,
           CASE WHEN robust_z > {T} THEN 'high' WHEN robust_z < -{T} THEN 'low' END AS direction,
           method = 'zero_spread' AND x <> med AS deviates_from_constant
    FROM robust_all
    WHERE scale = CASE WHEN parameter_std IN ({LOG_LIST}) THEN 'log10' ELSE 'linear' END
""")
TIMINGS["robust z"] = time.time() - t0
print(f"Robust statistics computed in {TIMINGS['robust z']:.0f} s")

### D.1 Flags by parameter

`evaluated` = rows with a z-score. **How to read it:** a few tenths of a percent flagged is typical for |z| > 5 on log-scale water-quality data; much higher shares point to mixed units or sites (section B), or to groups with low spread. `flagged_censored` are censored values far below the station's usual level (usually a very high detection limit, or a low one at a dirty site).

In [ ]:
robust_by_parameter = q("""
    SELECT parameter_std, any_value(scale) AS scale, count(*) AS rows,
           count(*) FILTER (WHERE method IN ('mad', 'meanad')) AS evaluated,
           count(*) FILTER (WHERE method = 'meanad') AS meanad_fallback,
           count(*) FILTER (WHERE method = 'sparse') AS sparse,
           count(*) FILTER (WHERE method = 'not_evaluable') AS not_evaluable,
           count(*) FILTER (WHERE method = 'zero_spread') AS zero_spread,
           count(*) FILTER (WHERE deviates_from_constant) AS deviates_from_constant,
           count(*) FILTER (WHERE direction = 'high') AS flagged_high,
           count(*) FILTER (WHERE direction = 'low') AS flagged_low,
           count(*) FILTER (WHERE flagged AND is_censored) AS flagged_censored,
           round(100 * count(*) FILTER (WHERE flagged) / nullif(count(*) FILTER (WHERE method IN ('mad', 'meanad')), 0), 2)
               AS pct_flagged_of_evaluated,
           count(DISTINCT station_id) FILTER (WHERE flagged) AS stations_with_flags
    FROM robust GROUP BY parameter_std ORDER BY rows DESC
""")
save(robust_by_parameter, "D1_robust_flags_by_parameter.csv")
robust_by_parameter

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
plot = robust_by_parameter.set_index("parameter_std")[["flagged_high", "flagged_low"]]
(100 * plot.div(robust_by_parameter.set_index("parameter_std")["evaluated"], axis=0)).plot.bar(stacked=True, ax=ax)
ax.set(ylabel="% of evaluated rows", title=f"Robust outlier flags, |z| > {T}", xlabel="")
plt.show()

### D.2 Flags by provider, season, station and years from removal

Years from removal come from `analysis_measurement`, so they count **pair rows** (a measurement used for two dams counts twice). **How to read it:** flags concentrated in one provider, season or a few stations point to a systematic cause; a jump right after year 0 could be a real removal effect (sediment pulse), which is one reason flags are not deletions.

In [ ]:
display(q("""
    SELECT provider, count(*) AS rows, count(*) FILTER (WHERE flagged) AS flagged,
           round(100 * avg(flagged::INT), 3) AS pct_flagged
    FROM robust WHERE method IN ('mad', 'meanad') GROUP BY provider
"""))
season = """CASE WHEN month IN (12, 1, 2) THEN 'winter' WHEN month IN (3, 4, 5) THEN 'spring'
               WHEN month IN (6, 7, 8) THEN 'summer' ELSE 'fall' END"""
display(q(f"""
    SELECT {season} AS season, count(*) AS rows, count(*) FILTER (WHERE flagged) AS flagged,
           round(100 * avg(flagged::INT), 3) AS pct_flagged
    FROM robust WHERE method IN ('mad', 'meanad') GROUP BY 1 ORDER BY 1
"""))
top_stations = q("""
    SELECT station_id, count(*) FILTER (WHERE flagged) AS flagged, count(*) AS rows,
           round(100 * avg(flagged::INT), 1) AS pct_flagged, string_agg(DISTINCT series, ', ') FILTER (WHERE flagged) AS series
    FROM robust WHERE method IN ('mad', 'meanad') GROUP BY station_id
    ORDER BY flagged DESC LIMIT 15
""")
save(top_stations, "D2_robust_top_stations.csv")
display(top_stations)
by_year = q("""
    SELECT a.years_from_removal, a.side, count(*) AS pair_rows, count(*) FILTER (WHERE r.flagged) AS flagged,
           round(100 * avg(r.flagged::INT), 3) AS pct_flagged
    FROM am a JOIN robust r USING (measurement_id)
    WHERE r.method IN ('mad', 'meanad') GROUP BY ALL ORDER BY a.side, a.years_from_removal
""")
save(by_year, "D2_robust_by_years_from_removal.csv")
fig, ax = plt.subplots(figsize=(8, 3.2))
sns.lineplot(data=by_year, x="years_from_removal", y="pct_flagged", hue="side", marker="o", ax=ax)
ax.axvline(0, color="k", ls="--", lw=1)
ax.set(title="Share of pair rows flagged, by year from removal", ylabel="% flagged")
plt.show()

### D.3 Threshold sensitivity, and log vs linear scale

**How to read it:** if the flagged share falls steeply from 3.5 to 5 and then flattens, the tail beyond 5 is a distinct group (likely errors); a smooth decline means the threshold is arbitrary for that parameter. The second table compares the two scales for log parameters: linear-scale screening flags mostly high values of skewed data (storm peaks), log-scale screening treats highs and lows symmetrically.

In [ ]:
sens = ", ".join(f"round(100 * avg((abs(robust_z) > {t})::INT), 3) AS \"pct_|z|>{t}\"" for t in THRESHOLDS["mad_sensitivity"])
sensitivity = q(f"SELECT parameter_std, {sens} FROM robust WHERE method IN ('mad', 'meanad') GROUP BY 1 ORDER BY 1")
save(sensitivity, "D3_threshold_sensitivity.csv")
display(sensitivity)
scale_compare = q(f"""
    WITH w AS (
        SELECT measurement_id, parameter_std,
               bool_or(scale = 'log10' AND abs(robust_z) > {T}) AS log_flag,
               bool_or(scale = 'linear' AND abs(robust_z) > {T}) AS linear_flag,
               bool_or(scale = 'linear' AND robust_z > {T}) AS linear_high
        FROM robust_all WHERE parameter_std IN ({LOG_LIST}) GROUP BY ALL
    )
    SELECT parameter_std, count(*) FILTER (WHERE log_flag) AS log_flags,
           count(*) FILTER (WHERE linear_flag) AS linear_flags,
           count(*) FILTER (WHERE log_flag AND linear_flag) AS both,
           count(*) FILTER (WHERE linear_flag AND NOT log_flag) AS linear_only,
           count(*) FILTER (WHERE log_flag AND NOT linear_flag) AS log_only,
           round(100 * count(*) FILTER (WHERE linear_high) / nullif(count(*) FILTER (WHERE linear_flag), 0), 1)
               AS pct_linear_flags_high
    FROM w GROUP BY 1 ORDER BY 1
""")
save(scale_compare, "D3_log_vs_linear.csv")
scale_compare

In [ ]:
# Saved: group statistics, and every row with |z| above the lowest sensitivity threshold
# (or deviating from a constant group). Source values and IDs are kept as they are.
lowest = min(THRESHOLDS["mad_sensitivity"])
save(q("SELECT DISTINCT station_id, series, scale, n_ref, med, mad, meanad FROM robust_all"),
     "D_robust_group_stats.parquet")
robust_flags_out = q(f"""
    SELECT measurement_id, station_id, series, parameter_std, scale, value, is_censored, robust_z, method,
           flagged, direction, deviates_from_constant
    FROM robust WHERE abs(robust_z) > {lowest} OR deviates_from_constant
""")
save(robust_flags_out, "D_robust_outlier_flags.parquet")
print(f"{len(robust_flags_out):,} rows saved with |z| > {lowest} or deviating from a constant group "
      f"({int(robust_flags_out['flagged'].sum()):,} flagged at |z| > {T})")

## E. Study eligibility and covariate coverage

Nothing here is a hard filter. Each issue is a **flag**; section I turns some of them into **hypothetical rules** and counts what each would remove.

### E.1 Site type: saline, tidal, estuarine, marsh or lake stations

Indicators for every station in `analysis_measurement`:
- `station_type` (WQP `MonitoringLocationTypeName`; the download already kept only the config's `station_types`);
- words in the station name suggesting a non-riverine water body (marsh, estuary, bay, tidal, lake, pond, reservoir, harbor, slough, lagoon, bayou...) or infrastructure (canal, ditch, outfall, effluent...);
- the NHDPlus feature type of the snapped reach. *Artificial path* is a flowline drawn through a lake, reservoir, estuary **or a wide river**, so on its own it is not evidence of a lake;
- station median specific conductance above `saline_sc_us_cm` (fresh water is usually well below 1,500 µS/cm; sea water is about 50,000);
- an NHDPlus drainage area of −9999 or null, common for coastal and network-terminal reaches.

In [ ]:
FTYPE = {460: "StreamRiver", 558: "ArtificialPath", 336: "CanalDitch", 566: "Coastline",
         334: "Connector", 420: "UndergroundConduit", 428: "Pipeline"}
WATERBODY_WORDS = r"\b(?:marsh|estuar\w*|bay|tidal|tide|lake|pond|reservoir|impoundment|harbou?r|inlet|sound|slough|lagoon|bayou|cove|pool|mouth)\b"
INFRA_WORDS = r"\b(?:canal|ditch|outfall|effluent|wwtp|stp|drain|discharge|spring|well)\b"
stream_cols = "r.gnis_name AS reach_name, r.feature_type, r.feature_code" if "streams" in HAS else \
    "NULL AS reach_name, NULL AS feature_type, NULL AS feature_code"
stream_join = "LEFT JOIN streams r ON r.nhdplus_id = s.NHDPlusID" if "streams" in HAS else ""
ws_cols = ("w.area_km2, w.nhdplus_total_area_km2, w.area_coverage, w.truncated" if "station_watershed" in HAS
           else "NULL AS area_km2, NULL AS nhdplus_total_area_km2, NULL AS area_coverage, NULL AS truncated")
ws_join = "LEFT JOIN station_watershed w USING (station_id)" if "station_watershed" in HAS else ""
station_flags = q(f"""
    WITH st AS (SELECT DISTINCT station_id FROM pairs),
    sc AS (SELECT station_id, median(result_value_std) AS sc_median, count(*) AS sc_n
           FROM m WHERE parameter_std = 'specific_conductance' GROUP BY 1)
    SELECT s.station_id, s.station_name, s.station_type, s.organization_id, s.huc8, s.state_code,
           s.snap_distance_m, s.NHDPlusID, {stream_cols}, sc.sc_median, sc.sc_n, {ws_cols}
    FROM st JOIN wq_station s USING (station_id) {stream_join}
    LEFT JOIN sc USING (station_id) {ws_join}
""")
names = station_flags["station_name"].fillna("").str.lower()
station_flags["name_waterbody_word"] = names.str.contains(WATERBODY_WORDS, regex=True)
station_flags["name_infrastructure_word"] = names.str.contains(INFRA_WORDS, regex=True)
station_flags["feature_label"] = station_flags["feature_type"].map(FTYPE).fillna(station_flags["feature_type"].astype(str))
station_flags["non_stream_feature"] = station_flags["feature_type"].isin([336, 566, 334, 420, 428])
station_flags["artificial_path"] = station_flags["feature_type"].eq(558)
station_flags["saline_conductance"] = station_flags["sc_median"] > THRESHOLDS["saline_sc_us_cm"]
station_flags["area_sentinel_or_null"] = ~(pd.to_numeric(station_flags["nhdplus_total_area_km2"], errors="coerce") > 0)
station_flags["truncated"] = station_flags["truncated"].fillna(False).astype(bool)
station_flags["no_watershed_row"] = station_flags["area_km2"].isna()
station_flags["snap_over_rule"] = station_flags["snap_distance_m"] > THRESHOLDS["snap_rule_m"]
station_flags["reach_name_match"] = [au.name_matches(s, r) for s, r in
                                     zip(station_flags["station_name"], station_flags["reach_name"])]
station_flags["name_mismatch"] = station_flags["reach_name_match"].eq(False)

print("station_type of analysis stations:")
display(station_flags["station_type"].value_counts(dropna=False).to_frame("stations"))
print("NHDPlus feature type of the snapped reach:")
display(station_flags["feature_label"].value_counts(dropna=False).to_frame("stations"))

In [ ]:
SITE_FLAGS = ["name_waterbody_word", "name_infrastructure_word", "non_stream_feature", "artificial_path",
              "saline_conductance", "area_sentinel_or_null", "no_watershed_row", "truncated",
              "snap_over_rule", "name_mismatch"]
con.register("station_flags_df", station_flags[["station_id"] + SITE_FLAGS])
con.execute("CREATE OR REPLACE TEMP TABLE station_flag_table AS SELECT * FROM station_flags_df")


def flag_impact(flag_table, key, flags):
    """For each boolean flag column: stations/pairs/dams/rows in am it touches."""
    out = []
    for f in flags:
        out.append(q(f"""
            SELECT '{f}' AS flag, count(DISTINCT a.station_id) AS stations, count(DISTINCT a.pair) AS pairs,
                   count(DISTINCT a.AR_ID) AS dams, count(*) AS rows
            FROM am a JOIN {flag_table} t USING ({key}) WHERE t.{f}
        """))
    return pd.concat(out, ignore_index=True)


site_impact = flag_impact("station_flag_table", "station_id", SITE_FLAGS)
site_impact["pct_rows"] = (100 * site_impact["rows"] / con.sql("SELECT count(*) FROM am").fetchone()[0]).round(2)
save(site_impact, "E1_site_flag_impact.csv")
save(station_flags, "E1_station_flags.parquet")
site_impact

In [ ]:
print("Examples: stations with a water-body word, saline conductance or a non-stream reach")
station_flags[station_flags["name_waterbody_word"] | station_flags["saline_conductance"] | station_flags["non_stream_feature"]][
    ["station_id", "station_name", "feature_label", "sc_median", "nhdplus_total_area_km2", "snap_distance_m"]
].sort_values("sc_median", ascending=False).head(20)

### E.2 Drainage areas, snapping and river names

The flags above, summarized: invalid drainage areas (`area_sentinel_or_null`, `no_watershed_row`, `truncated`), large snaps, and **name mismatches** (the distinctive words of the snapped reach's GNIS name don't appear in the station name, e.g. a station named "Mill Creek at ..." snapped to "Otter River"). A mismatch alone is weak evidence (names abbreviate, tributary sites are named after the main river), but **a mismatch plus a large snap distance** is a strong hint of a wrong reach. The same check is shown for dams (`River` vs the dam reach's GNIS name).

In [ ]:
sf = station_flags
snap_name = pd.crosstab(pd.cut(sf["snap_distance_m"], [-1, 100, 250, 500, 1000, 5000, np.inf],
                               labels=["<=100", "100-250", "250-500", "500-1k", "1k-5k", ">5k"]),
                        sf["reach_name_match"].map({True: "name matches", False: "name mismatch"}).fillna("no reach name"))
display(snap_name)
suspect = sf[sf["name_mismatch"] & sf["snap_over_rule"]]
print(f"Stations with a name mismatch AND snap > {THRESHOLDS['snap_rule_m']} m: {len(suspect)}")
display(suspect[["station_id", "station_name", "reach_name", "snap_distance_m"]].head(15))
if "streams" in HAS:
    dams_names = q("""
        SELECT d.AR_ID, d.Dam_Name, d.River, r.gnis_name AS reach_name, d.snap_distance_m
        FROM dam_removal d LEFT JOIN streams r ON r.nhdplus_id = d.NHDPlusID
        WHERE d.AR_ID IN (SELECT AR_ID FROM pairs)
    """)
    dams_names["match"] = [au.name_matches(river, reach) for river, reach in zip(dams_names["River"], dams_names["reach_name"])]
    print("Analysis dams: does the dam's River match its reach's GNIS name?")
    display(dams_names["match"].map({True: "matches", False: "mismatch"}).fillna("no name").value_counts().to_frame("dams"))
    save(dams_names, "E2_dam_river_name_check.csv")

### E.3 Dams: where they drop out

A funnel from every dam in `dam_removal` to the dams in `analysis_measurement`. Dams in the source CSV that never snapped to a reach are not in `dam_removal`, so they can't be counted here (they appear in the `snap_features` logs).

In [ ]:
dam_funnel = q("""
    SELECT 'in dam_removal' AS step, count(*) AS dams FROM dam_removal
    UNION ALL SELECT 'with a removal year', count(*) FROM dam_removal WHERE Year_Removed IS NOT NULL
    UNION ALL SELECT 'with a linked station', count(DISTINCT AR_ID) FROM dam_station_link
        WHERE AR_ID IN (SELECT AR_ID FROM dam_removal WHERE Year_Removed IS NOT NULL)
    UNION ALL SELECT 'with any pre/post measurement', count(DISTINCT AR_ID) FROM measurement_window WHERE period IN ('pre', 'post')
    UNION ALL SELECT 'with any has_both pair-series', count(DISTINCT AR_ID) FROM eligibility_pairs WHERE has_both
    UNION ALL SELECT 'with a downstream has_both pair-series', count(DISTINCT AR_ID) FROM eligibility_pairs
        WHERE has_both AND side = 'downstream'
    UNION ALL SELECT 'in analysis_measurement', count(DISTINCT AR_ID) FROM pairs
""")
save(dam_funnel, "E3_dam_funnel.csv")
dam_funnel

### E.4 Parameter and fraction coverage

One row per series in `analysis_measurement`. **How to read it:** series with fewer than `min_pairs_per_series` pairs (`small_series`) can't support a per-series model; high `pct_unit_basis_assumed` or many methods (B.3) mean the series mixes measurement conventions.

In [ ]:
method_cols = ("count(DISTINCT x.method_id) AS distinct_methods," if HAS_METHODS else "")
method_join = "LEFT JOIN m_method x USING (measurement_id)" if HAS_METHODS else ""
coverage = q(f"""
    SELECT a.series, count(*) AS rows, count(DISTINCT a.measurement_id) AS measurements,
           count(DISTINCT a.pair) AS pairs,
           count(DISTINCT a.pair) FILTER (WHERE a.side = 'downstream') AS downstream_pairs,
           count(DISTINCT a.pair) FILTER (WHERE a.side = 'upstream') AS upstream_pairs,
           count(DISTINCT a.AR_ID) AS dams, count(DISTINCT a.station_id) AS stations,
           round(100 * avg(a.is_censored::INT), 1) AS pct_censored,
           round(100 * avg(a.unit_basis_assumed::INT), 1) AS pct_unit_basis_assumed,
           round(100 * avg((a.provider = 'NWIS')::INT), 1) AS pct_nwis, {method_cols}
           string_agg(DISTINCT m.result_unit_raw, ', ') AS raw_units
    FROM am a JOIN m USING (measurement_id) {method_join}
    GROUP BY a.series ORDER BY rows DESC
""")
coverage["small_series"] = coverage["pairs"] < THRESHOLDS["min_pairs_per_series"]
save(coverage, "E4_parameter_fraction_coverage.csv")
coverage

### E.5 Covariate completeness

Per dam-station pair (climate is per row, so a pair's climate share is the share of its rows with climate). Kept **separate** from the water-quality checks: a pair can be fine for a water-quality-only model and unusable for one needing land cover.

- `land_cover_after_removal`: the baseline NLCD year is at or after removal, so the "pre-removal" predictor may already include the drained reservoir (spec 3.11, choice 2).
- `pre_1986_removal`: no NLCD year before removal exists (Annual NLCD starts in 1985).

In [ ]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE pair_cov AS
    SELECT AR_ID, station_id, pair, any_value(removal_year) AS removal_year, any_value(side) AS side, count(*) AS rows,
           avg((air_temp_mean_c IS NOT NULL)::INT) AS climate_share,
           avg((precip_prev_month_mm IS NOT NULL)::INT) AS prev_precip_share,
           any_value(land_cover_year) AS land_cover_year,
           bool_or(pct_forest IS NOT NULL) AS has_land_cover,
           any_value(upstream_area_km2) AS upstream_area_km2,
           coalesce(bool_or(watershed_truncated), FALSE) AS truncated,
           any_value(dam_height_ft) AS dam_height_ft, any_value(year_built) AS year_built,
           any_value(original_use) AS original_use
    FROM am GROUP BY AR_ID, station_id, pair
""")
pc = q("SELECT * FROM pair_cov")
pc["land_cover_after_removal"] = pc["land_cover_year"] >= pc["removal_year"]
pc["pre_1986_removal"] = pc["removal_year"] < 1986
checks = {
    "climate on all rows": pc["climate_share"] == 1,
    "climate on >= 95% of rows": pc["climate_share"] >= 0.95,
    "no climate at all": pc["climate_share"] == 0,
    "land cover present": pc["has_land_cover"],
    "land cover from before removal": pc["has_land_cover"] & ~pc["land_cover_after_removal"].fillna(True),
    "land cover at/after removal": pc["land_cover_after_removal"].fillna(False),
    "removal before 1986": pc["pre_1986_removal"],
    "upstream area present": pc["upstream_area_km2"].notna(),
    "watershed truncated": pc["truncated"],
    "dam height known": pc["dam_height_ft"].notna(),
    "year built known": pc["year_built"].notna(),
    "original use known": pc["original_use"].notna(),
}
covariates = pd.DataFrame([{
    "check": name, "pairs": int(mask.sum()), "pct_pairs": round(100 * mask.mean(), 1),
    "dams": pc.loc[mask, "AR_ID"].nunique(), "rows": int(pc.loc[mask, "rows"].sum()),
    "pct_rows": round(100 * pc.loc[mask, "rows"].sum() / pc["rows"].sum(), 1),
} for name, mask in checks.items()])
save(covariates, "E5_covariate_completeness.csv")
covariates

In [ ]:
# Low valid_fraction in the context tables: the covariate covers only part of the upstream area.
parts = []
if "climate" in HAS:
    parts.append(q("SELECT 'climate station-months' AS table_name, count(*) AS rows, "
                   "count(*) FILTER (WHERE valid_fraction < 0.9) AS valid_below_0_9, "
                   "count(*) FILTER (WHERE temperature_mean_c IS NULL) AS null_values FROM climate "
                   "WHERE station_id IN (SELECT station_id FROM pairs)"))
if "land_cover" in HAS:
    parts.append(q("SELECT 'land_cover station-years' AS table_name, count(*) AS rows, "
                   "count(*) FILTER (WHERE valid_fraction < 0.9) AS valid_below_0_9, "
                   "count(*) FILTER (WHERE percent_area IS NULL) AS null_values "
                   "FROM (SELECT station_id, year, any_value(valid_fraction) AS valid_fraction, "
                   "any_value(percent_area) AS percent_area FROM land_cover "
                   "WHERE station_id IN (SELECT station_id FROM pairs) GROUP BY ALL)"))
pd.concat(parts, ignore_index=True) if parts else "No climate or land_cover table"

### E.6 Temporal and seasonal coverage per pair

Per dam-station pair × series: samples, distinct dates and years on each side, months sampled on each side and how many are **shared** (a pre/post comparison where all pre samples are from summer and all post samples from winter mostly measures season), the gap between the last pre and first post sample, and whether the years right next to removal are covered.

In [ ]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE pair_time AS
    SELECT AR_ID, station_id, pair, series, parameter_std, any_value(side) AS side,
           any_value(removal_year) AS removal_year,
           count(*) FILTER (WHERE period = 'pre') AS n_pre,
           count(*) FILTER (WHERE period = 'post') AS n_post,
           count(DISTINCT sample_date) FILTER (WHERE period = 'pre') AS dates_pre,
           count(DISTINCT sample_date) FILTER (WHERE period = 'post') AS dates_post,
           count(DISTINCT year) FILTER (WHERE period = 'pre') AS years_pre,
           count(DISTINCT year) FILTER (WHERE period = 'post') AS years_post,
           count(DISTINCT month) FILTER (WHERE period = 'pre') AS months_pre,
           count(DISTINCT month) FILTER (WHERE period = 'post') AS months_post,
           coalesce(len(list_intersect(
               list(DISTINCT month) FILTER (WHERE period = 'pre'),
               list(DISTINCT month) FILTER (WHERE period = 'post'))), 0) AS shared_months,
           max(sample_date) FILTER (WHERE period = 'pre') AS last_pre,
           min(sample_date) FILTER (WHERE period = 'post') AS first_post,
           count(DISTINCT years_from_removal) FILTER (WHERE years_from_removal BETWEEN -2 AND -1) AS near_pre_years,
           count(DISTINCT years_from_removal) FILTER (WHERE years_from_removal BETWEEN 1 AND 2) AS near_post_years,
           bool_or(years_from_removal = -1) AS has_year_minus_1,
           bool_or(years_from_removal = 1) AS has_year_plus_1
    FROM am GROUP BY AR_ID, station_id, pair, series, parameter_std
""")
pair_time = q("SELECT *, date_diff('day', last_pre, first_post) AS gap_days FROM pair_time")
temporal_by_series = pair_time.groupby("series").agg(
    pair_series=("pair", "size"),
    median_n_pre=("n_pre", "median"), median_n_post=("n_post", "median"),
    median_dates_pre=("dates_pre", "median"), median_dates_post=("dates_post", "median"),
    median_years_pre=("years_pre", "median"), median_years_post=("years_post", "median"),
    median_shared_months=("shared_months", "median"),
    pct_shared_months_ge3=("shared_months", lambda s: round(100 * (s >= 3).mean(), 1)),
    pct_no_shared_month=("shared_months", lambda s: round(100 * (s == 0).mean(), 1)),
    median_gap_days=("gap_days", "median"),
).sort_values("pair_series", ascending=False)
both_sides = pair_time.assign(b=pair_time["has_year_minus_1"] & pair_time["has_year_plus_1"]).groupby("series")["b"].mean()
temporal_by_series["pct_years_minus1_and_plus1"] = (100 * both_sides).round(1)
save(pair_time, "E6_temporal_coverage_pairs.parquet")
save(temporal_by_series.reset_index(), "E6_temporal_coverage_by_series.csv")
temporal_by_series

In [ ]:
# Month sampled, pre vs post, as a share of each side's rows (largest 8 series).
months = q("""
    SELECT series, period, month, count(*) AS rows FROM am
    WHERE series IN (SELECT series FROM am GROUP BY series ORDER BY count(*) DESC LIMIT 8)
    GROUP BY ALL
""")
months["share"] = months["rows"] / months.groupby(["series", "period"])["rows"].transform("sum")
g = sns.catplot(data=months, x="month", y="share", hue="period", hue_order=["pre", "post"], col="series",
                col_wrap=4, kind="bar", height=2.4, aspect=1.3)
g.fig.suptitle("Sampling months, pre vs post", y=1.02)
plt.show()

### E.7 Hypothetical minimum-data rules

How many pair-series, pairs and dams **would** remain under stricter rules (configured in `THRESHOLDS["min_data_rules"]`). The first rule reproduces the current `has_both` definition (3 samples in 2 years per side), so its counts should equal the full table. **How to read it:** look for a rule that removes many thinly sampled pairs but few dams.

In [ ]:
def rule_sql(rule):
    terms = ["TRUE"]
    if "min_samples" in rule:
        terms.append(f"n_pre >= {rule['min_samples']} AND n_post >= {rule['min_samples']}")
    if "min_dates" in rule:
        terms.append(f"dates_pre >= {rule['min_dates']} AND dates_post >= {rule['min_dates']}")
    if "min_years" in rule:
        terms.append(f"years_pre >= {rule['min_years']} AND years_post >= {rule['min_years']}")
    if rule.get("min_shared_months"):
        terms.append(f"shared_months >= {rule['min_shared_months']}")
    if rule.get("near_removal"):
        terms.append("near_pre_years >= 1 AND near_post_years >= 1")
    return " AND ".join(terms)


overall, per_series = [], []
for rule in THRESHOLDS["min_data_rules"]:
    cond = rule_sql(rule)
    row = q(f"""
        SELECT count(*) AS rows, count(DISTINCT a.measurement_id) AS measurements,
               count(DISTINCT a.station_id) AS stations, count(DISTINCT a.pair) AS pairs,
               count(DISTINCT a.AR_ID) AS dams,
               count(DISTINCT a.AR_ID) FILTER (WHERE a.side = 'downstream') AS dams_with_downstream
        FROM am a JOIN pair_time t USING (pair, series) WHERE {cond}
    """).iloc[0].to_dict()
    overall.append({"rule": rule["rule"], **row})
    per_series.append(q(f"""
        SELECT '{rule["rule"]}' AS rule, series, count(*) AS pair_series, count(DISTINCT AR_ID) AS dams
        FROM pair_time WHERE {cond} GROUP BY series
    """))
min_data_overall = pd.DataFrame(overall)
min_data_series = pd.concat(per_series).pivot(index="series", columns="rule", values="pair_series").fillna(0).astype(int)
min_data_series = min_data_series[[r["rule"] for r in THRESHOLDS["min_data_rules"]]]
save(min_data_overall, "E7_min_data_rules_overall.csv")
save(min_data_series.reset_index(), "E7_min_data_rules_by_series.csv")
display(min_data_overall)
print("Pair-series kept, per series and rule:")
min_data_series.sort_values(min_data_series.columns[0], ascending=False)

## F. Overlapping exposures and upstream stations

### F.1 Overlapping removal windows

For each dam-station pair, the **other** removals linked to the same station (any relationship) whose removal year falls inside this pair's window (removal year − `years_before` … + `years_after`). **How to read it:** an overlapping removal means the station's pre or post period also contains another removal, so a change can't be attributed to one dam. `distance_ratio` compares the nearest overlapping removal's distance with this pair's own distance (a removal 10× farther away is probably a much weaker influence).

In [ ]:
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE pair_overlap AS
    WITH r AS (
        SELECT l.station_id, l.AR_ID, d.Year_Removed AS ry, l.relationship, l.network_distance_km AS d
        FROM dam_station_link l JOIN dam_removal d USING (AR_ID) WHERE d.Year_Removed IS NOT NULL
    )
    SELECT p.AR_ID, p.station_id, p.pair, p.removal_year, p.side, p.network_distance_km, p.rows,
           count(r.AR_ID) FILTER (WHERE r.ry BETWEEN p.removal_year - {YEARS_BEFORE} AND p.removal_year - 1) AS other_in_pre,
           count(r.AR_ID) FILTER (WHERE r.ry = p.removal_year) AS other_same_year,
           count(r.AR_ID) FILTER (WHERE r.ry BETWEEN p.removal_year + 1 AND p.removal_year + {YEARS_AFTER}) AS other_in_post,
           min(r.d) FILTER (WHERE r.ry BETWEEN p.removal_year - {YEARS_BEFORE} AND p.removal_year + {YEARS_AFTER}) AS nearest_other_km,
           string_agg(r.AR_ID, ', ') FILTER (WHERE r.ry BETWEEN p.removal_year - {YEARS_BEFORE} AND p.removal_year + {YEARS_AFTER}) AS other_ids
    FROM pairs p LEFT JOIN r ON r.station_id = p.station_id AND r.AR_ID <> p.AR_ID
    GROUP BY ALL
""")
po = q("SELECT * FROM pair_overlap")
po["overlap"] = (po["other_in_pre"] + po["other_same_year"] + po["other_in_post"]) > 0
po["distance_ratio"] = po["nearest_other_km"] / po["network_distance_km"].where(po["network_distance_km"] > 0)
overlap_summary = pd.DataFrame([{
    "pairs": len(po), "pairs_with_overlap": int(po["overlap"].sum()),
    "pct_pairs_with_overlap": round(100 * po["overlap"].mean(), 1),
    "rows_in_overlapping_pairs": int(po.loc[po["overlap"], "rows"].sum()),
    "pairs_overlap_in_pre": int((po["other_in_pre"] > 0).sum()),
    "pairs_overlap_same_year": int((po["other_same_year"] > 0).sum()),
    "pairs_overlap_in_post": int((po["other_in_post"] > 0).sum()),
    "stations_with_overlap": po.loc[po["overlap"], "station_id"].nunique(),
    "dams_with_any_overlapping_pair": po.loc[po["overlap"], "AR_ID"].nunique(),
    "dams_with_only_overlapping_pairs": int(po.groupby("AR_ID")["overlap"].all().sum()),
}]).T.rename(columns={0: "value"})
save(po, "F1_pair_overlap.parquet")
display(overlap_summary)
print("Overlapping pairs by side and number of other removals in the window:")
po.assign(others=(po["other_in_pre"] + po["other_same_year"] + po["other_in_post"]).clip(upper=5)) \
  .pivot_table(index="others", columns="side", values="pair", aggfunc="count", fill_value=0)

### F.2 Hypothetical ways to handle overlap (counted, not applied)

- **S1 exclude every overlapping pair**;
- **S2 nearest dam only**: each station keeps only its pair with the shortest network distance (any side);
- **S3 exclude only close overlaps**: drop a pair only if an overlapping removal is less than `overlap_distance_multiple` × as far as its own dam (or the distance is unknown).

**How to read it:** S2 keeps every station but loses dams seen only through shared stations; S1 keeps every dam that has at least one clean pair.

In [ ]:
mult = THRESHOLDS["overlap_distance_multiple"]
po["nearest_rank"] = po.sort_values(["network_distance_km", "AR_ID"]).groupby("station_id").cumcount()
scenarios = {
    "current (all pairs)": po["pair"],
    "S1 exclude overlapping pairs": po.loc[~po["overlap"], "pair"],
    "S2 nearest dam per station": po.loc[po["nearest_rank"] == 0, "pair"],
    f"S3 exclude overlaps closer than {mult:g}x": po.loc[~(po["overlap"] & ~(po["distance_ratio"] >= mult)), "pair"],
}
rows = []
for name, kept in scenarios.items():
    con.register("kept_pairs", pd.DataFrame({"pair": kept}))
    rows.append({"scenario": name, **au.counts(con, "SELECT a.* FROM am a JOIN kept_pairs USING (pair)")})
overlap_scenarios = pd.DataFrame(rows)
save(overlap_scenarios, "F2_overlap_scenarios.csv")
overlap_scenarios

### F.3 NHDPlus attributes for dam and station reaches (used by F.4 and G)

`dam_removal` and `wq_station` store the NHDPlus reach (`NHDPlusID`) but not its attributes. The slim NHDPlus GeoPackages already on disk keep the full `NHDPlusFlowlineVAA` table, so this reads, **read-only**, just the dam and station reaches: `TotDASqKm` (total drainage area at the reach's downstream end), `Slope` (m/m), and a few related fields. No network is rebuilt. Each package's VAA table is scanned whole (it has no index on `NHDPlusID`), so the **first run can take tens of minutes** on the `/courses` filesystem; progress is printed per package. The result is cached in the audit folder; later runs reuse it unless `REFRESH_VAA_CACHE` is set.

In [ ]:
VAA_CACHE = OUTPUT_DIR / "F3_nhdplus_vaa_subset.parquet"
VAA_COLUMNS = ["NHDPlusID", "TotDASqKm", "Slope", "SlopeLenKm", "MaxElevSmo", "MinElevSmo", "StreamOrde"]
vaa, vaa_notes = None, []
if RUN_NHDPLUS_VAA:
    if VAA_CACHE.exists() and not REFRESH_VAA_CACHE:
        vaa = pd.read_parquet(VAA_CACHE)
        print(f"Using cached VAA subset ({len(vaa):,} reaches): {VAA_CACHE}")
    else:
        t0 = time.time()
        ids = q("""SELECT NHDPlusID FROM dam_removal WHERE AR_ID IN (SELECT AR_ID FROM pairs)
                   UNION SELECT NHDPlusID FROM wq_station WHERE station_id IN (SELECT station_id FROM pairs)""")["NHDPlusID"]
        huc8s = q("""SELECT DISTINCT HUC8 AS huc8 FROM dam_removal WHERE AR_ID IN (SELECT AR_ID FROM pairs)
                     UNION SELECT DISTINCT huc8 FROM wq_station WHERE station_id IN (SELECT station_id FROM pairs)""")["huc8"]
        packages, no_package = set(), []
        for huc8 in sorted(huc8s.dropna()):
            try:
                packages.add(find_huc8_geodatabase(NHDPLUS_DIR, huc8))
            except FileNotFoundError:
                no_package.append(huc8)
        print(f"Reading NHDPlusFlowlineVAA for {len(ids):,} reaches from {len(packages)} packages "
              f"({len(no_package)} HUC8s without a package: {no_package[:10]})")
        vaa, vaa_notes = au.read_vaa(sorted(packages), ids, VAA_COLUMNS, progress=True)
        TIMINGS["NHDPlus VAA"] = time.time() - t0
        print(f"Found {len(vaa):,} of {len(ids):,} reaches in {TIMINGS['NHDPlus VAA']:.0f} s")
        if SAVE_OUTPUTS:
            au.save_output(vaa, VAA_CACHE.name, OUTPUT_DIR)
            SAVED.append(VAA_CACHE.name)
    for note in vaa_notes[:20]:
        print(" -", note)
    for column in ["totdasqkm", "slope", "slopelenkm", "maxelevsmo", "minelevsmo", "streamorde"]:
        if column not in vaa.columns:
            vaa[column] = np.nan
    con.register("vaa_df", vaa)
    con.execute("CREATE OR REPLACE TEMP TABLE vaa AS SELECT * FROM vaa_df")
    display(vaa.describe().T.round(4))
else:
    con.execute("CREATE OR REPLACE TEMP TABLE vaa (nhdplusid BIGINT, totdasqkm DOUBLE, slope DOUBLE)")
    print("Skipped: RUN_NHDPLUS_VAA is False. Dam drainage area and slope are not available without it.")

### F.4 Upstream stations: possibly within the former reservoir?

A **screening approximation**, not reservoir mapping. A dam of height *H* on a channel of slope *S* backs water up roughly *L ≈ H / S* (flat-pool approximation; real backwater curves can reach farther). An upstream station closer than *L* may have sat in the impounded reach, and is then itself affected by the removal (still water becoming river again), rather than a control.

- *H*: `Dam_Height_ft` × 0.3048 (missing for many dams).
- *S*: NHDPlus `Slope` of the **dam's** reach (inside the old pool, so possibly flattened, which makes *L* longer) and, for comparison, of the **station's** reach. Slopes ≤ 0 (including the −9998 sentinel) are treated as missing.
- Classes: `possibly within backwater` (distance ≤ L), `beyond estimate`, `no estimate`. Sensitivity columns use L/2 and 2L.

In [ ]:
upstream = q("""
    SELECT p.AR_ID, p.station_id, p.pair, p.network_distance_km, p.rows,
           d.Dam_Height_ft, d.Dam_Height_ft * 0.3048 AS height_m,
           CASE WHEN vd.slope > 0 THEN vd.slope END AS dam_reach_slope,
           CASE WHEN vs.slope > 0 THEN vs.slope END AS station_reach_slope
    FROM pairs p JOIN dam_removal d USING (AR_ID) JOIN wq_station s USING (station_id)
    LEFT JOIN vaa vd ON vd.nhdplusid = d.NHDPlusID
    LEFT JOIN vaa vs ON vs.nhdplusid = s.NHDPlusID
    WHERE p.side = 'upstream'
""")
upstream["backwater_km_dam_slope"] = upstream["height_m"] / upstream["dam_reach_slope"] / 1000
upstream["backwater_km_station_slope"] = upstream["height_m"] / upstream["station_reach_slope"] / 1000


def classify(distance, length, factor=1.0):
    return np.select([length.isna() | distance.isna(), distance <= factor * length],
                     ["no estimate", "possibly within backwater"], "beyond estimate")


upstream["class_dam_slope"] = classify(upstream["network_distance_km"], upstream["backwater_km_dam_slope"])
upstream["class_dam_slope_L_half"] = classify(upstream["network_distance_km"], upstream["backwater_km_dam_slope"], 0.5)
upstream["class_dam_slope_L_double"] = classify(upstream["network_distance_km"], upstream["backwater_km_dam_slope"], 2.0)
upstream["class_station_slope"] = classify(upstream["network_distance_km"], upstream["backwater_km_station_slope"])
save(upstream, "F4_upstream_backwater_screen.parquet")
summary = pd.concat({c: upstream[c].value_counts() for c in
                     ["class_dam_slope", "class_dam_slope_L_half", "class_dam_slope_L_double", "class_station_slope"]},
                    axis=1).fillna(0).astype(int)
print(f"Upstream pairs: {len(upstream)}; with dam height: {upstream['height_m'].notna().sum()}; "
      f"with dam-reach slope: {upstream['dam_reach_slope'].notna().sum()}")
display(summary)
upstream[["height_m", "dam_reach_slope", "station_reach_slope", "backwater_km_dam_slope", "network_distance_km"]].describe().round(4)

## G. Distance and drainage-area features (derived, exploratory)

### G.1 Network distance

`network_distance_km` is kept as it is. A log version is added for exploration: `log10(distance + offset)` with `offset = distance_log_offset_km` (0.1 km by default), so a 0 km distance (station on the dam's reach) stays defined; nulls stay null. No near/far cutoff is introduced.

In [ ]:
off = THRESHOLDS["distance_log_offset_km"]
dist = q("SELECT pair, side, relationship, network_distance_km FROM pairs")
dist["log10_distance_km"] = np.log10(dist["network_distance_km"] + off)
display(dist.groupby("side", dropna=False)["network_distance_km"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(2))
print(f"Null distances: {dist['network_distance_km'].isna().sum()}; zero distances: {(dist['network_distance_km'] == 0).sum()}")
fig, ax = plt.subplots(figsize=(8, 3.2))
sns.histplot(data=dist, x="log10_distance_km", hue="side", element="step", bins=50, ax=ax)
ax.set(xlabel=f"log10(network distance km + {off})", title="Dam-station network distance, one row per pair")
plt.show()

### G.2 Dam drainage area and the station / dam drainage-area ratio

Both areas come from the same source, NHDPlus `TotDASqKm` (F.3), so they are compatible: `drainage_area_ratio = station area / dam area`. It is a **proxy for watershed scale and potential tributary mixing**, not a measured dilution. Caveats: `TotDASqKm` is at each reach's downstream end, so for a dam partway along its reach the dam area is slightly overstated (worst for short distances and `same_reach` pairs). The station's warehouse upstream area (`station_watershed.area_km2`, computed by the pipeline) is shown for comparison.

`status` says why a ratio is missing or suspect: `dam reach not in VAA`, `station reach not in VAA`, `dam area invalid` / `station area invalid` (null, ≤ 0 or −9999), and `direction inconsistent` (a downstream station with a *smaller* area than the dam, or an upstream one with a larger area, beyond 1%).

In [ ]:
ws_cols = ("w.area_km2 AS station_area_km2_warehouse, w.nhdplus_total_area_km2 AS station_totda_warehouse"
           if "station_watershed" in HAS else "NULL AS station_area_km2_warehouse, NULL AS station_totda_warehouse")
ws_join = "LEFT JOIN station_watershed w ON w.station_id = p.station_id" if "station_watershed" in HAS else ""
spatial = q(f"""
    SELECT p.AR_ID, p.station_id, p.pair, p.side, p.relationship, p.network_distance_km, p.rows,
           d.NHDPlusID AS dam_reach, s.NHDPlusID AS station_reach,
           vd.nhdplusid IS NOT NULL AS dam_reach_in_vaa, vs.nhdplusid IS NOT NULL AS station_reach_in_vaa,
           vd.totdasqkm AS dam_totda_raw, vs.totdasqkm AS station_totda_raw,
           vd.package AS dam_package, vs.package AS station_package, {ws_cols}
    FROM pairs p JOIN dam_removal d USING (AR_ID) JOIN wq_station s ON s.station_id = p.station_id
    LEFT JOIN vaa vd ON vd.nhdplusid = d.NHDPlusID
    LEFT JOIN vaa vs ON vs.nhdplusid = s.NHDPlusID
    {ws_join}
""") if RUN_NHDPLUS_VAA else None
if spatial is not None:
    spatial["dam_area_km2"] = spatial["dam_totda_raw"].where(spatial["dam_totda_raw"] > 0)
    spatial["station_area_km2"] = spatial["station_totda_raw"].where(spatial["station_totda_raw"] > 0)
    spatial["drainage_area_ratio"] = spatial["station_area_km2"] / spatial["dam_area_km2"]
    spatial["log10_drainage_area_ratio"] = np.log10(spatial["drainage_area_ratio"])
    spatial["log10_distance_km"] = np.log10(spatial["network_distance_km"] + off)
    r = spatial["drainage_area_ratio"]
    spatial["status"] = np.select(
        [~spatial["dam_reach_in_vaa"], ~spatial["station_reach_in_vaa"],
         spatial["dam_area_km2"].isna(), spatial["station_area_km2"].isna(),
         ((spatial["side"] == "downstream") & (r < 0.99)) | ((spatial["side"] == "upstream") & (r > 1.01))],
        ["dam reach not in VAA", "station reach not in VAA", "dam area invalid", "station area invalid",
         "direction inconsistent"], "ok")
    spatial["provenance"] = "NHDPlusFlowlineVAA.TotDASqKm (reach outlet) from " + spatial["dam_package"].fillna("?")
    save(spatial, "G2_pair_spatial_features.parquet")
    display(pd.crosstab(spatial["status"], spatial["side"].fillna("(null)"), margins=True))
    display(spatial.groupby("side")["drainage_area_ratio"].describe(percentiles=[0.1, 0.5, 0.9]).round(2))
    agree = spatial.dropna(subset=["station_totda_warehouse"])
    agree = agree[agree["station_totda_warehouse"] > 0]
    print(f"Station TotDASqKm, VAA read here vs station_watershed: {np.isclose(agree['station_totda_raw'], agree['station_totda_warehouse'], rtol=1e-6).mean():.1%} identical "
          f"({len(agree)} pairs compared)")
else:
    print("Not available: needs the NHDPlus VAA step (RUN_NHDPLUS_VAA).")

In [ ]:
if spatial is not None:
    ok = spatial[spatial["status"] == "ok"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
    sns.histplot(data=ok, x="log10_drainage_area_ratio", hue="side", element="step", bins=50, ax=axes[0])
    axes[0].set(title="log10 station / dam drainage area", xlabel="log10 ratio (0 = same area)")
    sns.scatterplot(data=ok, x="log10_distance_km", y="log10_drainage_area_ratio", hue="side", s=10, alpha=0.5, ax=axes[1])
    axes[1].set(title="Distance vs drainage-area ratio", xlabel=f"log10(distance km + {off})", ylabel="log10 ratio")
    fig.tight_layout()
    plt.show()

## H. Sampling imbalance

Counts of `analysis_measurement` rows, measurements, stations, pairs and dams by one dimension at a time. Distance and drainage-area bins are **quintiles, for display only** (not proposed cutoffs). **How to read it:** compare `pct_rows` with `pct_pairs`: a category with many more rows than pairs is heavily sampled per pair, so a row-weighted model gives it more say than a pair- or dam-weighted one. Nothing is reweighted here.

In [ ]:
if spatial is not None:
    con.register("spatial_df", spatial[["pair", "drainage_area_ratio"]])
    ratio_join, ratio_col = "LEFT JOIN spatial_df sp USING (pair)", "sp.drainage_area_ratio"
else:
    ratio_join, ratio_col = "", "NULL::DOUBLE"
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE am_dims AS
    WITH pq AS (
        SELECT pair, ntile(5) OVER (ORDER BY network_distance_km) AS distance_quintile FROM pairs
        WHERE network_distance_km IS NOT NULL
    ),
    rq AS (
        SELECT pair, ntile(5) OVER (ORDER BY drainage_area_ratio) AS ratio_quintile
        FROM (SELECT pair, {ratio_col} AS drainage_area_ratio FROM pairs {ratio_join})
        WHERE drainage_area_ratio IS NOT NULL
    )
    SELECT a.measurement_id, a.AR_ID, a.station_id, a.pair, a.series, a.provider, a.side, a.period, a.season,
           a.state AS dam_state, substr(d.HUC8, 1, 2) AS huc2_region, d.HUC8 AS huc8,
           CAST(CAST(floor(a.removal_year / 10) * 10 AS INTEGER) AS VARCHAR) || 's' AS removal_decade,
           coalesce('Q' || CAST(pq.distance_quintile AS VARCHAR), 'no distance') AS distance_quintile,
           coalesce('Q' || CAST(rq.ratio_quintile AS VARCHAR), 'no ratio') AS drainage_ratio_quintile,
           CASE WHEN pc.climate_share >= 0.95 AND pc.has_land_cover AND pc.upstream_area_km2 IS NOT NULL
                     AND pc.dam_height_ft IS NOT NULL THEN 'complete' ELSE 'incomplete' END AS covariates
    FROM am a JOIN dam_removal d USING (AR_ID)
    LEFT JOIN pq USING (pair) LEFT JOIN rq USING (pair) LEFT JOIN pair_cov pc USING (pair)
""")
DIMENSIONS = ["series", "provider", "dam_state", "huc2_region", "side", "removal_decade", "season", "period",
              "distance_quintile", "drainage_ratio_quintile", "covariates"]
totals = q("SELECT count(*) AS rows, count(DISTINCT pair) AS pairs, count(DISTINCT AR_ID) AS dams, "
           "count(DISTINCT station_id) AS stations FROM am_dims").iloc[0]
imbalance = []
for dim in DIMENSIONS:
    t = q(f"""
        SELECT '{dim}' AS dimension, CAST({dim} AS VARCHAR) AS category, count(*) AS rows,
               count(DISTINCT measurement_id) AS measurements, count(DISTINCT station_id) AS stations,
               count(DISTINCT pair) AS pairs, count(DISTINCT AR_ID) AS dams
        FROM am_dims GROUP BY {dim}
    """)
    imbalance.append(t)
imbalance = pd.concat(imbalance, ignore_index=True)
imbalance["pct_rows"] = (100 * imbalance["rows"] / totals["rows"]).round(1)
imbalance["pct_pairs"] = (100 * imbalance["pairs"] / totals["pairs"]).round(1)
imbalance["pct_dams"] = (100 * imbalance["dams"] / totals["dams"]).round(1)
imbalance["rows_per_pair"] = (imbalance["rows"] / imbalance["pairs"]).round(0)
save(imbalance, "H_imbalance_by_dimension.csv")
print("pct_dams = share of dams with any rows in the category (a dam can be in several, so these don't sum to 100)")
imbalance[imbalance["dimension"].isin(["provider", "side", "period", "season", "removal_decade", "huc2_region", "covariates"])]

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, dim in zip(axes.flat, ["huc2_region", "removal_decade", "series", "provider", "distance_quintile", "season"]):
    t = imbalance[imbalance["dimension"] == dim].set_index("category")[["pct_rows", "pct_pairs", "pct_dams"]]
    t = t.sort_index() if dim not in ("series",) else t.sort_values("pct_rows", ascending=False)
    t.plot.bar(ax=ax, width=0.8, legend=ax is axes.flat[0])
    ax.set(title=dim, xlabel="", ylabel="%")
fig.tight_layout()
plt.show()

### H.1 Concentration: how much of the data comes from the most-sampled units

**How to read it:** if the top 1% of stations contribute, say, 30% of rows, a row-weighted model is largely a model of those stations.

In [ ]:
conc = []
for unit in ["station_id", "AR_ID", "pair"]:
    n = q(f"SELECT {unit} AS u, count(*) AS rows FROM am GROUP BY 1 ORDER BY rows DESC")
    share = n["rows"].cumsum() / n["rows"].sum()
    row = {"unit": unit, "units": len(n)}
    for top in [0.01, 0.05, 0.10, 0.25]:
        k = max(1, int(round(top * len(n))))
        row[f"top_{top:.0%}_units"] = k
        row[f"top_{top:.0%}_pct_rows"] = round(100 * share.iloc[k - 1], 1)
    conc.append(row)
    plt.plot(np.r_[0, np.arange(1, len(n) + 1) / len(n)], np.r_[0, share.values], label=unit)
concentration = pd.DataFrame(conc)
save(concentration, "H1_concentration.csv")
plt.plot([0, 1], [0, 1], color="gray", ls="--", lw=1)
plt.xlabel("share of units (most-sampled first)"); plt.ylabel("cumulative share of rows")
plt.title("Concentration of analysis rows"); plt.legend(); plt.show()
concentration

### H.2 Row-, pair- and dam-weighted summaries side by side

The same quantity summarized three ways: per row (as a pooled model sees it), averaged per pair, and averaged per dam. **How to read it:** large differences between the columns mean results will depend on the weighting choice, a decision for the modeling stage.

In [ ]:
weighted = q("""
    WITH r AS (SELECT a.*, (a.side = 'downstream')::INT AS downstream, (a.provider = 'NWIS')::INT AS nwis,
                      (a.period = 'post')::INT AS post, a.is_censored::INT AS censored,
                      (a.season = 'summer')::INT AS summer FROM am a)
    SELECT 'share downstream' AS quantity, avg(downstream) AS row_weighted,
           (SELECT avg(v) FROM (SELECT avg(downstream) AS v FROM r GROUP BY pair)) AS pair_weighted,
           (SELECT avg(v) FROM (SELECT avg(downstream) AS v FROM r GROUP BY AR_ID)) AS dam_weighted FROM r
    UNION ALL SELECT 'share NWIS', avg(nwis), (SELECT avg(v) FROM (SELECT avg(nwis) v FROM r GROUP BY pair)),
           (SELECT avg(v) FROM (SELECT avg(nwis) v FROM r GROUP BY AR_ID)) FROM r
    UNION ALL SELECT 'share post-removal', avg(post), (SELECT avg(v) FROM (SELECT avg(post) v FROM r GROUP BY pair)),
           (SELECT avg(v) FROM (SELECT avg(post) v FROM r GROUP BY AR_ID)) FROM r
    UNION ALL SELECT 'share censored', avg(censored), (SELECT avg(v) FROM (SELECT avg(censored) v FROM r GROUP BY pair)),
           (SELECT avg(v) FROM (SELECT avg(censored) v FROM r GROUP BY AR_ID)) FROM r
    UNION ALL SELECT 'share summer samples', avg(summer), (SELECT avg(v) FROM (SELECT avg(summer) v FROM r GROUP BY pair)),
           (SELECT avg(v) FROM (SELECT avg(summer) v FROM r GROUP BY AR_ID)) FROM r
    UNION ALL SELECT 'network distance (km)', avg(network_distance_km),
           (SELECT avg(v) FROM (SELECT avg(network_distance_km) v FROM r GROUP BY pair)),
           (SELECT avg(v) FROM (SELECT avg(network_distance_km) v FROM r GROUP BY AR_ID)) FROM r
""")
weighted[["row_weighted", "pair_weighted", "dam_weighted"]] = weighted[["row_weighted", "pair_weighted", "dam_weighted"]].round(3)
save(weighted, "H2_weighting_comparison.csv")
weighted

## I. Stepwise audit table and findings

### I.1 Flags gathered per analysis row

Every flag from sections B–G, joined onto each `analysis_measurement` row (`am_flags`, in memory). Row-level flags come from the measurement; pair- and station-level flags are copied to all rows of that pair or station.

In [ ]:
if spatial is not None:
    con.register("spatial_flags_df", spatial[["pair", "status"]])
    spatial_join = "LEFT JOIN spatial_flags_df sp ON sp.pair = a.pair"
    ratio_flag = "coalesce(sp.status = 'direction inconsistent', FALSE)"
else:
    spatial_join, ratio_flag = "", "FALSE"
small_series = sql_list(coverage.loc[coverage["small_series"], "series"]) or "''"
min_rule = next(r for r in THRESHOLDS["min_data_rules"] if r["rule"] == THRESHOLDS["rule_min_data"])
con.register("overlap_df", po[["pair", "overlap"]])
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE am_flags AS
    SELECT a.measurement_id, a.AR_ID, a.station_id, a.pair, a.series,
           -- measurement level
           a.measurement_id IN (SELECT measurement_id FROM value_flags) AS f_value_screen,
           a.measurement_id IN (SELECT measurement_id FROM unit_value_flags) AS f_unit_x1000,
           a.measurement_id IN (SELECT measurement_id FROM consistency_flags) AS f_consistency,
           coalesce(r.flagged, FALSE) AS f_robust_outlier,
           a.is_censored AS f_censored, a.unit_basis_assumed AS f_unit_basis_assumed,
           -- series / pair-series level
           a.series IN ({small_series}) AS f_small_series,
           coalesce(NOT ({rule_sql(min_rule)}), TRUE) AS f_min_data,
           coalesce(t.shared_months < {THRESHOLDS['rule_min_shared_months']}, TRUE) AS f_few_shared_months,
           -- pair level
           coalesce(o.overlap, FALSE) AS f_overlap,
           {ratio_flag} AS f_ratio_inconsistent,
           -- station level
           coalesce(s.saline_conductance, FALSE) AS f_saline,
           coalesce(s.name_waterbody_word OR s.non_stream_feature, FALSE) AS f_non_riverine_site,
           coalesce(s.snap_over_rule, FALSE) AS f_snap,
           coalesce(s.name_mismatch, FALSE) AS f_name_mismatch,
           coalesce(s.area_sentinel_or_null OR s.no_watershed_row OR s.truncated, FALSE) AS f_drainage_area,
           -- covariates
           a.air_temp_mean_c IS NULL OR a.precip_prev_month_mm IS NULL AS f_no_climate,
           a.pct_forest IS NULL OR a.land_cover_year >= a.removal_year AS f_no_pre_removal_land_cover,
           a.dam_height_ft IS NULL AS f_no_dam_height
    FROM am a
    LEFT JOIN robust r USING (measurement_id)
    LEFT JOIN pair_time t ON t.pair = a.pair AND t.series = a.series
    LEFT JOIN overlap_df o ON o.pair = a.pair
    LEFT JOIN station_flag_table s ON s.station_id = a.station_id
    {spatial_join}
""")
print(f"am_flags: {con.sql('SELECT count(*) FROM am_flags').fetchone()[0]:,} rows")

### I.2 Stepwise methods table

Two **hypothetical** sequences, kept separate as requested:

- **Sequence WQ** (water-quality analyses): measurement-quality rules, then series and minimum-data rules, then exposure and site rules.
- **Sequence COV** (analyses needing covariates): starts where WQ ends and adds the covariate rules.

For each step: rows the rule touches **on its own** (out of the full table), and what would **remain** after applying it **together with every step above it**. Listed at the bottom, but not applied, are flags that describe the data rather than proposing an exclusion (censoring, assumed unit basis, name mismatch, drainage-ratio direction). **None of these rules has been applied to any data.**

In [ ]:
STEPS = [
    ("WQ", "B.4 value screen (implausible zero / bound / beyond suspicious limit)", "f_value_screen", "measurement"),
    ("WQ", "B.5 value ~1000x its station median", "f_unit_x1000", "measurement"),
    ("WQ", "B.6 cross-parameter inconsistency", "f_consistency", "measurement"),
    ("WQ", f"D robust outlier |z| > {THRESHOLDS['mad_threshold']}", "f_robust_outlier", "measurement"),
    ("WQ", f"E.4 series with < {THRESHOLDS['min_pairs_per_series']} pairs", "f_small_series", "series"),
    ("WQ", f"E.7 fails '{THRESHOLDS['rule_min_data']}'", "f_min_data", "pair-series"),
    ("WQ", f"E.6 < {THRESHOLDS['rule_min_shared_months']} shared pre/post months", "f_few_shared_months", "pair-series"),
    ("WQ", "F.1 another removal inside the window", "f_overlap", "pair"),
    ("WQ", f"E.1 saline (median SC > {THRESHOLDS['saline_sc_us_cm']})", "f_saline", "station"),
    ("WQ", "E.1 non-riverine name or non-stream reach", "f_non_riverine_site", "station"),
    ("WQ", f"E.2 snap > {THRESHOLDS['snap_rule_m']} m", "f_snap", "station"),
    ("WQ", "E.2 drainage area invalid, missing or truncated", "f_drainage_area", "station"),
    ("COV", "E.5 no climate for the sample month", "f_no_climate", "measurement"),
    ("COV", "E.5 no land cover from before removal", "f_no_pre_removal_land_cover", "pair"),
    ("COV", "E.5 dam height unknown", "f_no_dam_height", "dam"),
]
INFO_FLAGS = [
    ("info", "censored value (kept; model censoring instead)", "f_censored", "measurement"),
    ("info", "nutrient unit basis assumed", "f_unit_basis_assumed", "measurement"),
    ("info", "station name doesn't match reach name", "f_name_mismatch", "station"),
    ("info", "drainage-area ratio direction inconsistent", "f_ratio_inconsistent", "pair"),
]
COUNT_SQL = ("count(*) AS rows, count(DISTINCT measurement_id) AS measurements, count(DISTINCT station_id) AS stations, "
             "count(DISTINCT pair) AS pairs, count(DISTINCT AR_ID) AS dams, count(DISTINCT series) AS series")
start = q(f"SELECT {COUNT_SQL} FROM am_flags").iloc[0].to_dict()
records = [{"sequence": "start", "step": "analysis_measurement as built", "kind": "-", "level": "-",
            "rows_flagged_alone": 0, **{f"remaining_{k}": v for k, v in start.items()}}]
applied = []
for sequence, label, column, level in STEPS:
    applied.append(column)
    alone = con.sql(f"SELECT count(*) FROM am_flags WHERE {column}").fetchone()[0]
    remaining = q(f"SELECT {COUNT_SQL} FROM am_flags WHERE NOT ({' OR '.join(applied)})").iloc[0].to_dict()
    records.append({"sequence": sequence, "step": label, "kind": "hypothetical exclusion", "level": level,
                    "rows_flagged_alone": alone, **{f"remaining_{k}": v for k, v in remaining.items()}})
for sequence, label, column, level in INFO_FLAGS:
    alone = con.sql(f"SELECT count(*) FROM am_flags WHERE {column}").fetchone()[0]
    records.append({"sequence": sequence, "step": label, "kind": "flag only (not applied)", "level": level,
                    "rows_flagged_alone": alone})
stepwise = pd.DataFrame(records)
stepwise["pct_rows_flagged_alone"] = (100 * stepwise["rows_flagged_alone"] / start["rows"]).round(2)
save(stepwise, "I2_stepwise_audit.csv")
stepwise

In [ ]:
# Independent impact: each rule applied alone (no ordering effects).
independent = pd.DataFrame([{
    "step": label, "level": level,
    **{f"remaining_{k}": v for k, v in q(f"SELECT {COUNT_SQL} FROM am_flags WHERE NOT {column}").iloc[0].to_dict().items()},
} for _, label, column, level in STEPS])
save(independent, "I2_rules_applied_alone.csv")
independent

### I.3 Findings summary

Generated from the tables above (numbers update on rerun) and saved as `findings_summary.md` with a `run_info.json` recording inputs, thresholds and outputs. The interpretation and the decisions still needed are in the next markdown cell.

In [ ]:
def pct(a, b):
    return f"{100 * a / b:.1f}%" if b else "n/a"


e = expansion.set_index("stage")
am_rows, am_meas = int(e.loc["5. analysis_measurement", "rows"]), int(e.loc["5. analysis_measurement", "measurements"])
wq_last = stepwise[stepwise["sequence"] == "WQ"].iloc[-1]
cov_last = stepwise[stepwise["sequence"] == "COV"].iloc[-1]
lines = [
    f"# Preprocessing audit findings: {cfg.name}",
    f"_Generated {datetime.now():%Y-%m-%d %H:%M} from {WAREHOUSE}; nothing in the warehouse was changed._",
    "",
    "## Data and joins",
    f"- analysis_measurement: {am_rows:,} rows from {am_meas:,} measurements "
    f"({am_rows / am_meas:.2f} rows per measurement), {start['stations']:,} stations, {start['pairs']:,} pairs, "
    f"{start['dams']:,} dams, {start['series']} series.",
    f"- Reused rows (same measurement, another dam): {int(reuse_check['reused_rows'][0]):,} "
    f"({pct(int(reuse_check['reused_rows'][0]), am_rows)}); inconsistent copies: {int(reuse_check['inconsistent_copies'][0])} "
    f"(0 = all repetition is reuse, not duplication).",
    f"- Pairs with another removal inside their window: {int(po['overlap'].sum()):,} of {len(po):,} "
    f"({pct(int(po['overlap'].sum()), len(po))}).",
    "",
    "## Measurement quality",
    f"- Value-screen flags (zeros, bounds, beyond suspicious limits): {int(value_flag_summary['rows'].sum()):,} flag rows.",
    f"- Unit-factor candidate blocks: {len(unit_candidates):,} ({int(unit_candidates['n'].sum()):,} rows); "
    f"single values ~1000x station median: {int(unit_value_summary['rows'].sum()):,}.",
    f"- Consistency violations (same activity): " + "; ".join(
        f"{r.check_name}: {r.violations:,}/{r.comparisons:,}" for r in
        consistency_summary[consistency_summary['basis'] == 'same_activity'].itertuples()) + ".",
    f"- Robust outliers (|z| > {T}): {int(robust_by_parameter['flagged_high'].sum() + robust_by_parameter['flagged_low'].sum()):,} "
    f"of {int(robust_by_parameter['evaluated'].sum()):,} evaluated rows.",
    "",
    "## Duplicates and repeated sampling",
    "- " + "; ".join(f"{r.check_name}: {int(r.groups or 0):,} groups" for r in dup_checks.itertuples()) + ".",
    f"- Co-located station pairs with matching values: {int(colocated_summary['station_pairs_with_matches'][0]):,} "
    f"({int(colocated_summary['exact_matches'][0]):,} exact matches).",
    f"- Rows on days with several readings: {int(same_day['rows_on_multi_days'].fillna(0).sum()):,} of "
    f"{int(same_day['rows'].sum()):,} cleaned results.",
    "",
    "## Eligibility, covariates and imbalance",
    f"- Hypothetical WQ sequence would keep {int(wq_last['remaining_rows']):,} rows / {int(wq_last['remaining_pairs']):,} pairs / "
    f"{int(wq_last['remaining_dams']):,} dams; adding covariate rules: {int(cov_last['remaining_rows']):,} rows / "
    f"{int(cov_last['remaining_pairs']):,} pairs / {int(cov_last['remaining_dams']):,} dams.",
    f"- Top 1% of stations ({concentration.set_index('unit').loc['station_id', 'top_1%_units']}) contribute "
    f"{concentration.set_index('unit').loc['station_id', 'top_1%_pct_rows']}% of rows.",
    "- Small series: " + (", ".join(coverage.loc[coverage['small_series'], 'series']) or "none") + ".",
]
if spatial is not None:
    lines.append(f"- Drainage-area ratio available for {int((spatial['status'] == 'ok').sum()):,} of {len(spatial):,} pairs; "
                 + ", ".join(f"{k}: {v}" for k, v in spatial['status'].value_counts().items()) + ".")
if RUN_NHDPLUS_VAA:
    lines.append(f"- Upstream backwater screen (dam-reach slope): "
                 + ", ".join(f"{k}: {v}" for k, v in upstream['class_dam_slope'].value_counts().items()) + ".")
findings = "\n".join(lines) + "\n"
if SAVE_OUTPUTS:
    (OUTPUT_DIR / "findings_summary.md").write_text(findings)
    SAVED.append("findings_summary.md")
display(Markdown(findings))

In [ ]:
inputs = {view: {"path": str(WAREHOUSE / f"{view}.parquet"),
                 "modified": datetime.fromtimestamp((WAREHOUSE / f"{view}.parquet").stat().st_mtime).isoformat()}
          for view in VIEWS}
run_info = {
    "generated": datetime.now().isoformat(timespec="seconds"),
    "case_study": cfg.name, "warehouse": str(WAREHOUSE), "output_dir": str(OUTPUT_DIR),
    "window": {"years_before": YEARS_BEFORE, "years_after": YEARS_AFTER},
    "current_eligibility": {"min_samples": MIN_SAMPLES, "min_years": MIN_YEARS},
    "switches": {"RUN_METHOD_AUDIT": RUN_METHOD_AUDIT, "RUN_NHDPLUS_VAA": RUN_NHDPLUS_VAA},
    "thresholds": THRESHOLDS, "log_parameters": LOG_PARAMETERS,
    "suspicious_limits": SUSPICIOUS_LIMITS, "zero_rules": ZERO_RULES,
    "duckdb_version": duckdb.__version__, "inputs": inputs,
    "timings_s": {k: round(v, 1) for k, v in TIMINGS.items()},
    "outputs": sorted(set(SAVED)),
}
if SAVE_OUTPUTS:
    (OUTPUT_DIR / "run_info.json").write_text(json.dumps(run_info, indent=2, default=str))
print(f"{len(set(SAVED))} outputs in {OUTPUT_DIR}")
print("Timings (s):", run_info["timings_s"])

### I.4 Interpretation and decisions needed before model-specific preparation

_Fill in after reviewing a run. Suggested headings:_

- **Major limitations found** (with the table that shows each).
- **Rules worth adopting**, with their cost from I.2 (rows / pairs / dams lost).
- **Decisions that need approval** before the model-specific stage:
  1. which measurement-quality flags become exclusions (B.4–B.6, D), and the robust-z threshold;
  2. whether to aggregate same-day readings (C.3) and how (daily median, or per depth);
  3. how to treat measurement reuse and overlapping removals (A.4, F.1–F.2: cluster by station, nearest dam, or exclude close overlaps);
  4. which site rules apply (saline, non-riverine, snap distance, drainage area: E.1–E.2);
  5. the minimum-data rule (E.7) and seasonal-overlap requirement (E.6);
  6. how to use distance, the drainage-area ratio and the backwater screen (G, F.4) as features;
  7. weighting or aggregation level for imbalance (H), separately for the mixed-effects and predictive models.